# Meta-Analysis of GWAS Results: Combining Two Batches of *Hypolimnas misippus* with GWAMA and METAL

**Course:** *Genome-Wide Association Studies (GWAS) for Evolutionary Biology* (prstats.org)  
**Topic:** Meta-analysis of GWAS results. Combining association results across studies with **GWAMA** and **METAL**; fixed-effect and random-effect models; heterogeneity assessment.

This notebook **continues** the alignment notebook (`QC_Alignment_Hypolimnas_misippus_Colab.ipynb`), which produced the BAM files in the `bams/` folder of your Google Drive. Starting from those BAM files we will

1. **split the 20 butterflies into two "batches" (two mini-studies)** of 10 individuals each, 5 *m*-morphs and 5 *i*-morphs per batch;
2. **call variants with ANGSD separately in each batch** (as two independent studies would do);
3. **test association with the wing-pattern morph (*m* vs *i*) separately in each batch** with ANGSD `-doAsso`, and draw the two Manhattan plots side by side;
4. turn each ANGSD result into a **summary-statistics file** (effect allele, effect size, standard error, p-value, sample size), which is the *only* thing a meta-analysis needs;
5. **harmonise** the alleles of the two studies and run the **meta-analysis** three ways: **our own Python implementation** (so you can see every formula), **GWAMA**, and **METAL**; fixed-effect (inverse-variance), random-effects (DerSimonian-Laird), and the sample-size-weighted *Z*-score method;
6. **assess heterogeneity** (Cochran's *Q*, *I*², τ²) and **consistency of effect directions** between the batches;
7. produce the final **Manhattan plot and QQ plot of the meta-analysis**, a **table of the top-ranked variants**, and **forest plots**.

> **Honest framing.** Two random halves of the *same* sample are not two independent cohorts, and 10 individuals per batch is very small. The aim is to learn the *workflow and the logic* on data you know well (the M locus on chromosome 26 is a built-in positive control). The same code works unchanged for real studies with hundreds or thousands of individuals; only the input files change.

Everything runs on Google Colab: the notebook installs its own software (Conda, ANGSD, GWAMA, METAL), mounts your Google Drive, and writes all results back to Drive.

## 0. Background: why, and how, do we combine GWAS?

### 0.1 Why meta-analysis?

A single GWAS has limited **power**: a variant with a modest effect needs thousands of individuals to reach genome-wide significance. Instead of waiting for one giant study, the field combines the **summary statistics** of many studies. This has three practical advantages:

* **Power.** Adding studies adds information. Combining two studies of equal size has (for a homogeneous effect) the power of one study with twice the individuals.
* **No sharing of individual-level data.** Genotypes of people (or of protected species, or of samples under a restrictive agreement) cannot always be pooled; summary statistics usually can.
* **Replication and robustness.** A true association should point **in the same direction** in independent samples. A signal driven by one batch, one sequencing run or one population stratum usually does not. Meta-analysis gives us a formal way to quantify this (heterogeneity).

### 0.2 What is a "study" and what do we need from it?

For every variant, each study reports a handful of numbers. Nothing else is needed:

| Quantity | Symbol | Meaning |
|---|---|---|
| Marker identifier | | Chromosome and position (and alleles) so that variants can be matched between studies |
| **Effect allele** (EA) and **other allele** (NEA) | | The allele whose effect is estimated, and the allele it is compared with |
| **Effect size** | $\hat\beta_i$ | Change in log-odds of the phenotype per copy of the effect allele (binary trait) or change in the trait (quantitative trait) in study $i$ |
| **Standard error** | $\mathrm{SE}_i$ | Precision of $\hat\beta_i$ |
| P-value | $P_i$ | Used by methods that do not need effect sizes |
| Sample size | $N_i$ | Used for weighting in the sample-size scheme |
| Effect-allele frequency | $f_i$ | Used for quality control (do the studies look like the same population?) |

### 0.3 The fixed-effect (inverse-variance weighted) model

The **fixed-effect** model assumes that there is **one true effect** $\beta$ shared by all studies, and that the studies differ only by sampling noise:

$$\hat\beta_i \sim N(\beta,\ \mathrm{SE}_i^2).$$

The best (minimum-variance) combined estimate weights each study by its **precision**, $w_i = 1/\mathrm{SE}_i^2$:

$$\hat\beta_{FE}=\frac{\sum_i w_i\,\hat\beta_i}{\sum_i w_i},\qquad \mathrm{SE}_{FE}=\frac{1}{\sqrt{\sum_i w_i}},\qquad Z_{FE}=\frac{\hat\beta_{FE}}{\mathrm{SE}_{FE}},\qquad P_{FE}=2\,\Phi(-|Z_{FE}|).$$

Large, precise studies count more. This is what **METAL** calls `SCHEME STDERR` and what **GWAMA** computes by default.

### 0.4 Heterogeneity: do the studies agree?

If the fixed-effect assumption is true, the studies' estimates scatter around $\hat\beta_{FE}$ only by their standard errors. **Cochran's Q** measures the excess scatter:

$$Q=\sum_i w_i\,(\hat\beta_i-\hat\beta_{FE})^2 \ \sim\ \chi^2_{K-1}\quad\text{under homogeneity},$$

where $K$ is the number of studies. A small p-value of $Q$ means that the studies disagree more than chance allows. Because $Q$ grows with the number of studies and with their precision, we also report

$$I^2=\max\!\Big(0,\ \frac{Q-(K-1)}{Q}\Big)\times 100\%,$$

the **percentage of the total variation that is due to real heterogeneity and not to chance** (0–25 % low, 25–50 % moderate, 50–75 % substantial, > 75 % considerable). 

> **Warning for $K=2$.** With only two studies, $Q$ has **one** degree of freedom and very little power; a non-significant $Q$ does not prove that effects are homogeneous. $I^2$ is also very unstable. This is a general limitation of small meta-analyses, which we will meet in our own data.

### 0.5 The random-effects model

The **random-effects** model admits that the true effect may **differ between studies** (different populations, environments, sequencing, phenotyping, ...). Each study has its own true effect, drawn from a distribution with mean $\beta$ and between-study variance $\tau^2$:

$$\hat\beta_i \sim N(\beta_i,\ \mathrm{SE}_i^2),\qquad \beta_i \sim N(\beta,\ \tau^2).$$

The **DerSimonian-Laird** estimator of $\tau^2$ is

$$\hat\tau^2=\max\!\Big(0,\ \frac{Q-(K-1)}{\sum_i w_i-\sum_i w_i^2/\sum_i w_i}\Big),$$

and the random-effects weights are $w_i^{*}=1/(\mathrm{SE}_i^2+\hat\tau^2)$, giving $\hat\beta_{RE}=\sum_i w_i^{*}\hat\beta_i/\sum_i w_i^{*}$ with $\mathrm{SE}_{RE}=1/\sqrt{\sum_i w_i^{*}}$.

What changes? When $\hat\tau^2=0$ (no excess heterogeneity), the random-effects result is identical to the fixed-effect result. When there is heterogeneity, $\mathrm{SE}_{RE}>\mathrm{SE}_{FE}$: confidence intervals are **wider**, p-values are **less extreme**, and the weights of the studies become **more equal** (small studies get relatively more weight). The price is lower power. GWAMA reports random-effects results with the option `--random`.

*Which model should I use?* If studies are similar (same population, same design, like our two batches), the fixed-effect model is the standard; in GWAS, the fixed-effect model is the primary analysis, and heterogeneity statistics are used to flag loci to inspect.

### 0.6 The sample-size-weighted *Z*-score method (METAL `SCHEME SAMPLESIZE`)

Sometimes effect sizes and standard errors are not comparable or not available (different models, different scales). Then we can combine only the **p-values and directions** of effect. Each study gives a signed $Z_i=\mathrm{sign}(\hat\beta_i)\,\Phi^{-1}(1-P_i/2)$. They are combined by **Stouffer's method with weights** $\sqrt{N_i}$:

$$Z_{SS}=\frac{\sum_i \sqrt{N_i}\,Z_i}{\sqrt{\sum_i N_i}}.$$

It gives no combined effect size, but it is robust to scale differences between studies. If all studies have similar SEs per sample, it is nearly equivalent to the inverse-variance method.

### 0.7 Harmonising alleles: the most common source of silent errors

Study 1 may report "effect allele = A, other = G", study 2 "effect allele = G, other = A" for the *same* SNP. Adding their effect sizes without checking would **cancel real effects**. Before combining we must:

* match variants by chromosome and position;
* check that the two alleles are the same set;
* if the effect alleles are swapped, **flip** study 2 (multiply $\hat\beta$ by $-1$, and the frequency becomes $1-f$);
* drop variants whose alleles do not match (tri-allelic sites, or genotype errors). For human data, A/T and C/G SNPs are additionally strand-ambiguous; ANGSD reports alleles on the reference strand, so this is not an issue here.

GWAMA and METAL do this internally (they align every study to the effect allele of the first one they see), and we will do the same explicitly in Python so that we can check that the tools and our code agree. ANGSD chooses "major/minor" from the data of each batch, so **flips will occur**, especially at sites with allele frequency near 0.5.

### 0.8 GWAMA and METAL in one table

| | **GWAMA** (Mägi & Morris 2010) | **METAL** (Willer et al. 2010) |
|---|---|---|
| Input | one file per study, with named columns (marker, effect allele, other allele, beta/SE or OR/CI, N, ...) plus a *file list* | one file per study, columns declared in a *command script* |
| Fixed effect | yes (inverse variance) | `SCHEME STDERR` |
| Random effects | yes (`--random`) | not in the standard run |
| Sample-size / p-value scheme | no | `SCHEME SAMPLESIZE` |
| Heterogeneity | *Q*, *I*² by default | `ANALYZE HETEROGENEITY` (*Q*, *I*², p) |
| Genomic control | `--genomic_control` | `GENOMICCONTROL ON` |
| Allele alignment | automatic (flips and checks) | automatic |
| Extras | sex-specific and trans-ethnic analyses | tracks effect direction per study (`+-?`) |

### 0.9 Genomic control and other safeguards

Each study may be **inflated** (population structure, relatedness, batch effects). Before the meta-analysis, a common safeguard is to divide each study's $\chi^2$ statistics by its genomic-control factor $\lambda_{GC}=\mathrm{median}(\chi^2)/0.4549$ if $\lambda_{GC}>1$. Both tools can do it (`-gc`, `GENOMICCONTROL ON`); our notebook has a switch. Do not use it blindly: with a **true polygenic or large-effect signal** (like the M locus) or with very few SNPs, the median is not a pure null estimate.

### 0.10 Pitfalls checklist

1. **Sample overlap** between studies makes them not independent: *Z*-scores are correlated and p-values anti-conservative. *(Our two batches do not overlap; each butterfly is in exactly one batch.)*
2. **Different models/covariates** (and different allele coding) across studies. We use the same ANGSD model in both batches.
3. **Low coverage**: genotype uncertainty differs between studies if sequencing depth differs; this can create *spurious heterogeneity*. We check depth balance between the batches in Section 5.
4. **Allele misalignment** (Section 0.7).
5. **Unbalanced precision**: a large study dominates. Look at the weights in the forest plot.
6. **Heterogeneity tests with few studies** have little power (Section 0.4).
7. **Winner's curse**: the top hits of a discovery study are over-estimated; effects in an independent study are usually smaller. We examine this in Section 11 (discovery in one batch, "replication" in the other).

### 0.11 Experimental design of this notebook

```
 bams/*.sorted.bam (20 butterflies, from the alignment notebook)
        │
        │  stratified random split: 5 m + 5 i  |  5 m + 5 i
        ▼
  ┌──────────── Batch B1 ─────────────┐   ┌──────────── Batch B2 ─────────────┐
  │ ANGSD: genotype likelihoods,      │   │ ANGSD: genotype likelihoods,      │
  │        SNP calling                │   │        SNP calling                │
  │ ANGSD -doAsso 2 (m vs i)          │   │ ANGSD -doAsso 2 (m vs i)          │
  └───────────────┬───────────────────┘   └───────────────┬───────────────────┘
                  │ summary statistics (EA, NEA, beta, SE, P, N, freq)
                  └───────────────────┬───────────────────┘
                                      ▼
              harmonise alleles → META-ANALYSIS: Python, GWAMA, METAL
                                      ▼
        Manhattan + QQ, top variants, direction consistency, forest plots
```

## 1. Install Conda in Google Colab

Google Colab does not ship with Conda. The `condacolab` package installs a minimal Conda (Miniforge) into the running session. **The kernel restarts automatically at the end of the next cell** (Colab may show a message that the session crashed; this is expected). After the restart, continue with Section 2 *without* re-running this cell.

In [ ]:
# Install Conda (Miniforge) inside Google Colab. The kernel restarts automatically afterwards.
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. Install ANGSD, SAMtools, GWAMA and METAL

| Tool | What we use it for | How it is installed |
|---|---|---|
| **ANGSD** | genotype likelihoods, SNP calling and association testing per batch | Bioconda |
| **SAMtools** | quick integrity check of the BAM files | Bioconda |
| **GWAMA** | fixed- and random-effects meta-analysis, heterogeneity | `apt` (Ubuntu package `gwama`), fallback Bioconda |
| **METAL** | `STDERR` and `SAMPLESIZE` meta-analysis, heterogeneity | Conda (channel `dnachun`), fallback: compiled from the source code on GitHub |

If an installation fails, the notebook **still runs**: our own Python implementation of the meta-analysis (Section 8) does not need GWAMA or METAL; the tools are used for cross-validation. You will see a clear message in Section 9.

In [ ]:
%%bash
# ANGSD, SAMtools and friends from Bioconda (+ conda-forge for the dependencies).
mamba install -y -q -c conda-forge -c bioconda angsd samtools bcftools htslib 2>&1 | tail -n 5

In [ ]:
%%bash
# GWAMA: the Ubuntu package is the most reliable route; fall back to Bioconda.
if ! command -v GWAMA >/dev/null 2>&1 && ! command -v gwama >/dev/null 2>&1; then
    (apt-get update -qq && apt-get install -y -qq gwama) > /tmp/apt_gwama.log 2>&1 || tail -n 3 /tmp/apt_gwama.log
fi
if ! command -v GWAMA >/dev/null 2>&1 && ! command -v gwama >/dev/null 2>&1; then
    echo "apt route failed; trying Bioconda ..."
    mamba install -y -q -c conda-forge -c bioconda gwama 2>&1 | tail -n 3 || true
fi
command -v GWAMA || command -v gwama || echo "GWAMA NOT FOUND (the notebook will skip the GWAMA part)"

In [ ]:
%%bash
# METAL: try the Conda package first; if that fails compile it from the official source code.
if ! command -v metal >/dev/null 2>&1; then
    mamba install -y -q -c conda-forge -c dnachun metal 2>&1 | tail -n 3 || true
fi
if ! command -v metal >/dev/null 2>&1; then
    echo "Conda route failed; compiling METAL from source ..."
    (apt-get install -y -qq cmake zlib1g-dev > /dev/null 2>&1 || true)
    cd /content && rm -rf METAL_src && git clone -q --depth 1 https://github.com/statgen/METAL.git METAL_src \
      && mkdir -p METAL_src/build && cd METAL_src/build \
      && cmake -DCMAKE_BUILD_TYPE=Release .. > /tmp/metal_cmake.log 2>&1 \
      && make -j4 > /tmp/metal_make.log 2>&1 \
      && BIN=$(find /content/METAL_src -name metal -type f -perm -u+x | head -n 1) \
      && cp "$BIN" /usr/local/bin/metal || tail -n 5 /tmp/metal_make.log
fi
command -v metal || echo "METAL NOT FOUND (the notebook will skip the METAL part)"

### 2.1 Verify the installation

Recording the exact version of every program is part of reproducible science.

In [ ]:
%%bash
echo "ANGSD:    $(angsd 2>&1 | grep -i -m1 'version')"
echo "samtools: $(samtools --version | head -n 1)"
echo "GWAMA:    $( (GWAMA --version 2>&1 || gwama --version 2>&1) | head -n 1 )   [$(command -v GWAMA || command -v gwama)]"
echo "METAL:    $(command -v metal)"
echo "QUIT" | metal 2>&1 | grep -i -m1 "version" || true

## 3. Mount Google Drive and Locate the BAM Files

Colab's local disk is wiped when the session ends, so all inputs and outputs live on **Google Drive**. Running the next cell asks for permission to access your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3.1 Locate the project and define all parameters

The next cell

* searches your Drive for the `bams/` folder written by the alignment notebook (folder with `*.sorted.bam` files; the project folder is its parent);
* creates the output folder `metaanalysis_HypMis/` next to it;
* defines **every parameter of the analysis in one place**. Change a value here and re-run the cells that depend on it.

In [ ]:
import os, re, json, gzip, glob, shutil, subprocess, io, time, itertools, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

# ---------------------------------------------------------------- where are the BAM files?
DRIVE_ROOT = Path("/content/drive/MyDrive")
BAM_FOLDER_NAME = "bams"
# If the automatic search fails or picks the wrong folder, set the path here, e.g.
# BAM_DIR_OVERRIDE = "/content/drive/MyDrive/GWAS_course/bams"
BAM_DIR_OVERRIDE = None

def find_on_drive(name, kind, maxdepth=6):
    # Search Google Drive for a folder (kind='d') or file (kind='f') with an exact name.
    res = subprocess.run(["find", str(DRIVE_ROOT), "-maxdepth", str(maxdepth),
                          "-type", kind, "-name", name], capture_output=True, text=True)
    return [Path(p) for p in res.stdout.splitlines() if p.strip()]

if BAM_DIR_OVERRIDE:
    BAM_DIR = Path(BAM_DIR_OVERRIDE)
else:
    hits = [p for p in find_on_drive(BAM_FOLDER_NAME, "d") if any(p.glob("*.sorted.bam"))]
    assert hits, (f"Could not find a folder called '{BAM_FOLDER_NAME}' with *.sorted.bam files in your Google Drive. "
                  "Run the alignment notebook first, or set BAM_DIR_OVERRIDE above.")
    hits.sort(key=lambda p: (not (p.parent / "phenotypes.tsv").exists(), str(p)))
    if len(hits) > 1:
        print("WARNING: several matching folders found, using the first one:")
        for h in hits: print("   ", h)
    BAM_DIR = hits[0]
assert BAM_DIR.is_dir(), f"{BAM_DIR} is not a directory"
PROJECT_DIR = BAM_DIR.parent

def locate_file(name):
    p = PROJECT_DIR / name
    if p.exists():
        return p
    hits = find_on_drive(name, "f")
    return hits[0] if hits else None

# ---------------------------------------------------------------- output folders (on Drive = persistent)
OUT = PROJECT_DIR / "metaanalysis_HypMis"
D = {k: OUT / k for k in ["angsd", "summary_stats", "gwama", "metal", "tables", "figures"]}
WORK = Path("/content/work_meta")             # local disk: fast but temporary
LOCAL_BAMS = WORK / "bams"
for d in list(D.values()) + [WORK]:
    d.mkdir(parents=True, exist_ok=True)

# ================================================================ ANALYSIS PARAMETERS
THREADS = os.cpu_count() or 2
FORCE = False                 # True = redo every step even if its output exists
COPY_BAMS_LOCAL = True        # copy BAMs from Drive to the local disk first (much faster for ANGSD)

# ---- the two batches (Section 5)
BATCH_NAMES = ["B1", "B2"]
N_PER_MORPH_PER_BATCH = 5     # 5 m + 5 i = 10 individuals per batch
SPLIT_SEED = 42               # the random split is reproducible
SPLIT_MODE = "random"         # "random" = stratified random split;  "balance_depth" = best of 2000 random splits for equal sequencing depth

# ---- read and site filters for ANGSD (applied identically in both batches)
MIN_MAPQ = 30                 # discard reads with mapping quality below this
MIN_BASEQ = 20                # discard bases with Phred quality below this
GL_MODEL = 2                  # 1 = SAMtools GL, 2 = GATK GL
SNP_PVAL = 1e-6               # keep a site as a SNP only if the likelihood-ratio test of monomorphism has P below this
MIN_MAF = 0.10                # minimum minor allele frequency (10 individuals = 20 chromosomes: 0.10 means >= 2 copies)
MIN_IND = None                # minimum individuals with data at a site; None = chosen from a depth pilot in each batch (Section 6)
MAX_DEPTH = None              # maximum total depth at a site; None = chosen from the depth pilot in each batch

# ---- phenotype
CASE_MORPH = "m"              # this morph is coded 1 ("case"), the other morph 0 ("control")

# ---- ANGSD association (Section 7)
ASSO_MODE = 2                 # 2 = score test (fast, dosage-based). The LRT statistic is chi-square(1) under the null.
ASSO_MIN_HIGH = 0             # ANGSD -minHigh (0 = off: with low coverage confident genotypes are rare)
ASSO_MIN_COUNT = 5            # ANGSD -minCount: minimum expected minor-allele count (= MAF x 2N; 5 of 20 chromosomes = MAF 0.25)

# ---- meta-analysis (Sections 8-9)
EFFECT_SOURCE = "angsd_lrt"   # "angsd_lrt": signed Z from the ANGSD LRT (default);  "score": Z recomputed by us from the genotype dosages
APPLY_GC = False              # divide each batch's chi-square by its genomic-control lambda (if > 1) before combining
RUN_GWAMA = True
RUN_METAL = True
GENOME_WIDE_P = 5e-8          # the conventional human genome-wide threshold (shown for reference); we also use Bonferroni
TOP_N = 20                    # number of top variants in the table; forest plots show the top 6

# ---- the M locus (positive control)
PEAK_START, PEAK_END = 6_731_000, 6_743_400
INCLUDE_Z = False             # females are ZW: the Z has one copy per individual, so we analyse autosomes only

# ---- colours
MORPH_ORDER = ["m", "i"]
MORPH_COLORS = {"m": "#4c72b0", "i": "#dd8452"}
BATCH_COLORS = {"B1": "#0072B2", "B2": "#D55E00"}
META_COLORS = {"fixed": "#009E73", "random": "#CC79A7", "ss": "#E69F00"}
CHUNK = 200_000               # rows processed at a time when reading the big ANGSD files (lower it if Colab runs out of memory)
KEYMULT = 10**9               # marker key = chromosome index * 1e9 + position (see Section 7.4)

# ---------------------------------------------------------------- make paths visible to %%bash cells
for k, v in {"PROJECT_DIR": PROJECT_DIR, "BAM_DIR": BAM_DIR, "OUT": OUT, "WORK": WORK, "THREADS": THREADS}.items():
    os.environ[k] = str(v)

print("Project folder :", PROJECT_DIR)
print("BAM folder     :", BAM_DIR, f"({len(list(BAM_DIR.glob('*.sorted.bam')))} BAM files)")
print("Output folder  :", OUT)
print("CPU threads    :", THREADS)
print("GWAMA          :", shutil.which("GWAMA") or shutil.which("gwama") or "NOT FOUND (see Section 2)")
print("METAL          :", shutil.which("metal") or "NOT FOUND (see Section 2)")

In [ ]:
# A small helper that runs a shell command, keeps the log on Drive and shows the end of the log.
def run(cmd, log=None, check=True, show=12, quiet=False):
    if not quiet:
        print("$", cmd if len(cmd) < 600 else cmd[:600] + " ...")
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable="/bin/bash")
    text = (r.stdout or "") + (r.stderr or "")
    if log is not None:
        Path(log).write_text(f"$ {cmd}\n\n{text}")
    if r.returncode != 0 and check:
        print(text[-3500:])
        raise RuntimeError(f"Command failed with exit code {r.returncode} (see the output above).")
    if show and text.strip() and not quiet:
        lines = text.strip().splitlines()
        print("\n".join(lines[-show:]))
    if not quiet:
        print(f"[done in {time.time() - t0:.1f} s, exit code {r.returncode}]")
    return r

def done(*paths):
    # True if every file exists and is non-empty (and FORCE is off): used to make steps resumable.
    return (not FORCE) and all(Path(p).exists() and Path(p).stat().st_size > 0 for p in paths)

def savefig(name):
    plt.savefig(D["figures"] / f"{name}.png", dpi=150, bbox_inches="tight")

### 3.2 The sample table: BAM files, morphs and the binary phenotype

The alignment notebook wrote `phenotypes.tsv` (sample ID, morph, depth). If it is missing we rebuild the table from the BAM names and the morph annotation. The phenotype is binary: **`m` = 1 ("case"), `i` = 0 ("control")**, as in the paper.

In [ ]:
pheno_path = PROJECT_DIR / "phenotypes.tsv"
if pheno_path.exists():
    pheno = pd.read_csv(pheno_path, sep="\t")
    print("Read", pheno_path, "->", len(pheno), "samples")
else:
    print("phenotypes.tsv not found: rebuilding the sample table from the BAM names and the morph annotation.")
    morph_file = locate_file("Hypolimnas_misippus_CAMID_morph.txt")
    assert morph_file is not None, "Hypolimnas_misippus_CAMID_morph.txt not found: upload it to the project folder."
    morph = pd.read_csv(morph_file, sep="\t"); morph.columns = ["sampleID", "morph"]
    bams = sorted(BAM_DIR.glob("*.sorted.bam"))
    pheno = pd.DataFrame({"sampleID": [b.name.replace(".sorted.bam", "") for b in bams]}).merge(morph, on="sampleID")

pheno = pheno.copy()
pheno["bam"] = pheno.sampleID.map(lambda s: str(BAM_DIR / f"{s}.sorted.bam"))
pheno["y"] = (pheno.morph == CASE_MORPH).astype(int)
missing_bam = [b for b in pheno.bam if not Path(b).exists()]
missing_bai = [b for b in pheno.bam if Path(b).exists() and not Path(b + ".bai").exists()]
assert not missing_bam, f"{len(missing_bam)} BAM files are missing, e.g. {missing_bam[:3]}"
assert not missing_bai, f"{len(missing_bai)} BAM files have no .bai index, e.g. {missing_bai[:3]}"
assert pheno.sampleID.is_unique, "duplicated sample IDs"
assert set(pheno.morph) <= {"m", "i"}, "unexpected morph labels"
need = 2 * N_PER_MORPH_PER_BATCH
assert (pheno.morph == "m").sum() >= need and (pheno.morph == "i").sum() >= need, (
    f"Need at least {need} m and {need} i individuals for two batches with {N_PER_MORPH_PER_BATCH} of each morph per batch.")
print(f"\n{len(pheno)} samples with BAM + index.  Morphs: {pheno.morph.value_counts().to_dict()}")
display(pheno.drop(columns=["bam"]).head())

### 3.3 Integrity check and local copy of the BAM files

`samtools quickcheck` reads the header and the end-of-file marker of every BAM; it detects truncated files in a second. ANGSD reads every BAM many times (twice per batch plus the depth pilot), so copying the BAM files from Drive to Colab's local disk first is much faster. This is only sensible for small BAM files; set `COPY_BAMS_LOCAL = False` for large ones.

In [ ]:
r = subprocess.run(["samtools", "quickcheck", "-v"] + list(pheno.bam), capture_output=True, text=True)
bad = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]
assert not bad, f"samtools quickcheck found problems in {len(bad)} BAM file(s): {bad[:3]}"
print("All BAM files pass samtools quickcheck.")

if COPY_BAMS_LOCAL:
    LOCAL_BAMS.mkdir(parents=True, exist_ok=True)
    total = sum(Path(b).stat().st_size + Path(b + ".bai").stat().st_size for b in pheno.bam)
    free = shutil.disk_usage(WORK).free
    if total * 1.2 < free:
        for b in pheno.bam:
            for ext in ("", ".bai"):
                dst = LOCAL_BAMS / (Path(b).name + ext)
                if not dst.exists() or dst.stat().st_size != Path(b + ext).stat().st_size:
                    shutil.copy(b + ext, dst)
        pheno["bam"] = [str(LOCAL_BAMS / Path(b).name) for b in pheno.bam]
        print(f"Copied {total/1e9:.2f} GB of BAM files to the local disk; ANGSD will read them from {LOCAL_BAMS}")
    else:
        print(f"Not enough local disk ({free/1e9:.1f} GB free for {total/1e9:.1f} GB): ANGSD will read from Drive.")
else:
    print("COPY_BAMS_LOCAL = False: ANGSD will read the BAM files from Drive.")

## 4. The Meta-Analysis Methods in Python, and a Simulation to Build Intuition

Before touching real data we implement the formulas of Section 0 in a few lines of NumPy, **vectorised over variants** (one row per study, one column per variant). We will use exactly these functions on the real data later, and compare them with GWAMA and METAL.

### 4.1 The functions

* `meta_analysis(beta, se)` returns, for every variant, the fixed-effect estimate, its SE, *Z* and P; Cochran's *Q* with its P; $I^2$; $\hat\tau^2$ (DerSimonian-Laird); and the random-effects estimate, SE, *Z* and P. Studies with a missing estimate (`NaN`) are ignored for that variant.
* `stouffer(z, n)` is the sample-size-weighted *Z*-score method.
* `forest_panel(...)` draws a forest plot: one square per study (area proportional to its weight, horizontal line = 95 % confidence interval) and diamonds for the pooled estimates.

In [ ]:
def meta_analysis(beta, se):
    # Fixed-effect (inverse-variance) and random-effects (DerSimonian-Laird) meta-analysis.
    # beta, se : arrays of shape (K studies, M variants); NaN = study does not have this variant.
    beta = np.atleast_2d(np.asarray(beta, dtype=np.float64)); se = np.atleast_2d(np.asarray(se, dtype=np.float64))
    ok = np.isfinite(beta) & np.isfinite(se) & (se > 0)
    w = np.where(ok, 1.0 / np.where(ok, se, 1.0) ** 2, 0.0)           # precision weights; 0 for missing studies
    b = np.where(ok, beta, 0.0)
    K = ok.sum(0)
    with np.errstate(divide="ignore", invalid="ignore"):
        sw = w.sum(0)
        b_fe = (w * b).sum(0) / sw
        se_fe = 1.0 / np.sqrt(sw)
        z_fe = b_fe / se_fe
        Q = (w * (b - b_fe) ** 2).sum(0)                               # Cochran's Q
        df = K - 1
        p_Q = np.where(df > 0, stats.chi2.sf(Q, np.maximum(df, 1)), np.nan)
        I2 = np.where(df > 0, np.where(Q > 0, np.maximum(0.0, (Q - df) / Q), 0.0), np.nan)
        denom = sw - (w ** 2).sum(0) / sw
        tau2 = np.where(df > 0, np.maximum(0.0, (Q - df) / denom), np.nan)
        wr = np.where(ok, 1.0 / (np.where(ok, se, 1.0) ** 2 + np.nan_to_num(tau2)), 0.0)   # random-effects weights
        swr = wr.sum(0)
        b_re = (wr * b).sum(0) / swr
        se_re = 1.0 / np.sqrt(swr)
        z_re = b_re / se_re
    return dict(K=K, beta_fe=b_fe, se_fe=se_fe, z_fe=z_fe, p_fe=2 * stats.norm.sf(np.abs(z_fe)),
                Q=Q, p_Q=p_Q, I2=I2, tau2=tau2,
                beta_re=b_re, se_re=se_re, z_re=z_re, p_re=2 * stats.norm.sf(np.abs(z_re)),
                w_fe=w / sw, w_re=wr / swr)

def stouffer(z, n):
    # Sample-size weighted Z-score (METAL's SCHEME SAMPLESIZE): sum(sqrt(N_i) Z_i) / sqrt(sum N_i).
    z = np.atleast_2d(np.asarray(z, dtype=np.float64)); n = np.atleast_2d(np.asarray(n, dtype=np.float64))
    ok = np.isfinite(z) & np.isfinite(n) & (n > 0)
    zz = np.where(ok, z, 0.0); nn = np.where(ok, n, 0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        Z = (np.sqrt(nn) * zz).sum(0) / np.sqrt(nn.sum(0))
    return Z, 2 * stats.norm.sf(np.abs(Z))

def lambda_gc(p):
    # Genomic inflation factor: median of the observed chi-square statistics / median of chi2 with 1 d.f. (0.4549)
    p = np.asarray(p, dtype=np.float64)
    p = p[np.isfinite(p) & (p > 0) & (p <= 1)]
    return float(np.median(stats.chi2.isf(p, 1)) / stats.chi2.ppf(0.5, 1)) if len(p) else np.nan

def forest_panel(ax, labels, beta, se, fe=None, re=None, title="", log_or=True, study_colors=None, show_re=True):
    # Forest plot. beta/se per study; fe/re = (beta, se) of the pooled estimates (diamonds).
    beta = np.asarray(beta, float); se = np.asarray(se, float)
    f = np.exp if log_or else (lambda x: x)
    n_rows = len(labels) + (fe is not None) + (re is not None and show_re)
    ypos = np.arange(n_rows)[::-1].astype(float)
    w = 1.0 / se ** 2; sizes = 40 + 260 * w / np.nanmax(w)
    ylabels = []
    for i, (lab, b, s) in enumerate(zip(labels, beta, se)):
        if not (np.isfinite(b) and np.isfinite(s)):
            ylabels.append(f"{lab} (no data)"); continue
        col = (study_colors or {}).get(lab, "0.25")
        ax.plot([f(b - 1.96 * s), f(b + 1.96 * s)], [ypos[i]] * 2, color="k", lw=1.3, zorder=2)
        ax.scatter([f(b)], [ypos[i]], s=sizes[i], marker="s", color=col, zorder=3)
        ylabels.append(lab)
    k = len(labels)
    for est, key, col, name in ((fe, "fe", META_COLORS["fixed"], "fixed effect"), (re, "re", META_COLORS["random"], "random effects")):
        if est is None or (key == "re" and not show_re): continue
        row = k if key == "fe" else k + (fe is not None)
        y = ypos[row]; b, s = est
        ax.fill([f(b - 1.96 * s), f(b), f(b + 1.96 * s), f(b)], [y, y + 0.32, y, y - 0.32], color=col, zorder=3)
        ylabels.append(name)
    ax.axvline(1.0 if log_or else 0.0, color="grey", ls="--", lw=1)
    ax.set_yticks(ypos[:len(ylabels)]); ax.set_yticklabels(ylabels, fontsize=8)
    if log_or: ax.set_xscale("log")
    ax.set_xlabel("odds ratio per effect allele (log scale)" if log_or else "effect size (beta)", fontsize=8)
    ax.set_title(title, fontsize=9); ax.tick_params(axis="x", labelsize=8)

# ---- unit test against a hand calculation:  beta = (0.5, 0.3), SE = (0.1, 0.2)
t = meta_analysis([[0.5], [0.3]], [[0.1], [0.2]])
assert abs(t["beta_fe"][0] - 0.46) < 1e-12 and abs(t["se_fe"][0] - 1 / np.sqrt(125)) < 1e-12
assert abs(t["Q"][0] - 0.8) < 1e-12 and abs(t["p_Q"][0] - stats.chi2.sf(0.8, 1)) < 1e-12
print("hand check passed:  w = (100, 25) -> beta_FE = (100*0.5 + 25*0.3)/125 = 0.46,  SE_FE = 1/sqrt(125) = 0.0894,  Q = 0.16 + 0.64 = 0.80")
print(f"random effects for this example: tau2 = {t['tau2'][0]:.4f} (Q < K-1 = 1, so tau2 = 0 and RE = FE)")

### 4.2 Why bother? The power gain from combining studies

For a true effect $\beta$, a study with standard error $\mathrm{SE}$ has $Z\sim N(\beta/\mathrm{SE},1)$. Combining $K$ equally sized studies reduces the SE to $\mathrm{SE}/\sqrt K$ (Section 0.3). The curves show the probability of reaching the usual human genome-wide threshold $P<5\times10^{-8}$ ($|Z|>5.45$) as a function of the true log odds ratio. The shift to the left is the *entire* benefit of meta-analysis in the homogeneous case.

In [ ]:
se1 = 0.30                                         # standard error of one study (log-odds scale)
zthr = stats.norm.isf(GENOME_WIDE_P / 2)
beta_grid = np.linspace(0, 2.5, 200)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for K_, col in zip([1, 2, 4, 8], ["#999999", "#0072B2", "#009E73", "#CC79A7"]):
    ncp = beta_grid / (se1 / np.sqrt(K_))
    power = stats.norm.sf(zthr - ncp) + stats.norm.cdf(-zthr - ncp)
    ax[0].plot(beta_grid, power, color=col, lw=2, label=f"{K_} stud{'y' if K_ == 1 else 'ies'} (SE = {se1/np.sqrt(K_):.2f})")
ax[0].set(xlabel="true log odds ratio", ylabel=f"power at P < {GENOME_WIDE_P:g}", title="Power gain from meta-analysis (equal-sized studies)"); ax[0].legend()
ax[1].axis("off")
ax[1].text(0, 0.95, "How to read this\n\n"
           "• One study with SE = 0.30 needs a true log-OR of\n  about 1.6 to reach 80 % power.\n"
           "• Two such studies reach 80 % power with ~1.15,\n  four with ~0.8: the effect that can be detected\n  shrinks as 1/sqrt(K).\n"
           "• The same holds for sample size: pooling two\n  studies of N each is like one study of 2N\n  (homogeneous effects, no overlap).",
           va="top", fontsize=10)
plt.tight_layout(); savefig("sim_power"); plt.show()

### 4.3 Simulation: when do fixed and random effects differ, and what does heterogeneity look like?

We simulate $K=4$ studies of different precision ($\mathrm{SE}_i = 0.25, 0.30, 0.35, 0.40$) for 20,000 independent "variants" in **four scenarios**:

| Scenario | True effect in study $i$ | What it represents |
|---|---|---|
| **A. no effect, homogeneous** | $\beta_i = 0$ | the vast majority of the genome |
| **B. effect, homogeneous** | $\beta_i = 0.5$ | an ideal causal variant |
| **C. effect, heterogeneous** | $\beta_i\sim N(0.5,\,0.3^2)$ | a causal variant whose effect depends on the study |
| **D. no mean effect, heterogeneous** | $\beta_i\sim N(0,\,0.3^2)$ | a variant with study-specific effects that average to zero |

For each scenario we report how often each method gives $P<0.001$, how often $Q$ is significant ($P_Q<0.05$), the median $I^2$ and the mean $\hat\tau^2$.

In [ ]:
rng = np.random.default_rng(7)
K, M = 4, 20000
se_true = np.array([0.25, 0.30, 0.35, 0.40])[:, None] * np.ones((1, M))
scen = {"A  no effect, homogeneous": np.zeros((K, M)),
        "B  effect 0.5, homogeneous": np.full((K, M), 0.5),
        "C  effect 0.5, heterogeneous (tau=0.3)": 0.5 + 0.3 * rng.standard_normal((K, M)),
        "D  no mean effect, heterogeneous (tau=0.3)": 0.3 * rng.standard_normal((K, M))}
rows, store, est_store = [], {}, {}
for name, btrue in scen.items():
    est = btrue + se_true * rng.standard_normal((K, M))
    r = meta_analysis(est, se_true); store[name] = r; est_store[name] = est
    rows.append({"scenario": name, "P_FE < 0.001": (r["p_fe"] < 1e-3).mean(), "P_RE < 0.001": (r["p_re"] < 1e-3).mean(),
                 "P_Q < 0.05": (r["p_Q"] < 0.05).mean(), "median I2 (%)": 100 * np.median(r["I2"]), "mean tau2": r["tau2"].mean()})
sim_tab = pd.DataFrame(rows).set_index("scenario")
display(sim_tab.style.format({"P_FE < 0.001": "{:.4f}", "P_RE < 0.001": "{:.4f}", "P_Q < 0.05": "{:.3f}", "median I2 (%)": "{:.1f}", "mean tau2": "{:.3f}"}))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for (name, r), col in zip(store.items(), ["#999999", "#0072B2", "#009E73", "#CC79A7"]):
    ax[0].hist(r["p_Q"], bins=20, histtype="step", lw=2, color=col, label=name[:1], density=True)
ax[0].set(xlabel="P of Cochran's Q", ylabel="density", title="Heterogeneity test\n(uniform if the studies agree)"); ax[0].legend(title="scenario")
rD = store["D  no mean effect, heterogeneous (tau=0.3)"]
for key, col, lab in (("p_fe", META_COLORS["fixed"], "fixed effect"), ("p_re", META_COLORS["random"], "random effects")):
    p = np.sort(rD[key]); e = (np.arange(1, len(p) + 1) - 0.5) / len(p)
    ax[1].plot(-np.log10(e), -np.log10(np.maximum(p, 1e-300)), color=col, lw=2, label=lab)
ax[1].plot([0, 4.3], [0, 4.3], "k--", lw=1); ax[1].set(xlabel="expected -log10 P", ylabel="observed -log10 P", title="Scenario D (a NULL on average):\nboth are inflated, fixed effect more so"); ax[1].legend()
nmC = "C  effect 0.5, heterogeneous (tau=0.3)"
vi = int(np.argmin(np.abs(store[nmC]["I2"] - 0.6)))                 # an illustrative variant of scenario C with I2 close to 60 %
eC = est_store[nmC][:, vi]
rr = meta_analysis(eC[:, None], se_true[:, :1])
forest_panel(ax[2], [f"study {i+1}" for i in range(K)], eC, se_true[:, 0], fe=(rr["beta_fe"][0], rr["se_fe"][0]), re=(rr["beta_re"][0], rr["se_re"][0]),
             title=f"One heterogeneous variant: Q P={rr['p_Q'][0]:.3f}, I2={100*rr['I2'][0]:.0f}%, tau2={rr['tau2'][0]:.3f}", log_or=False)
plt.tight_layout(); savefig("sim_heterogeneity"); plt.show()

**What the simulation teaches.**

* **Scenario A.** Both models have the nominal false-positive rate (0.1 %), $P_Q$ is uniform, $I^2$ is small.
* **Scenario B.** Fixed and random effects are the same and very powerful.
* **Scenario C.** $Q$ now flags many variants and $I^2$ is substantial. The fixed-effect model is still more powerful than the random-effects model, which pays for the heterogeneity with wider intervals.
* **Scenario D** is the cautionary tale. The fixed-effect test **assumes there is one effect**; when study-specific effects scatter around zero it produces far too many false positives (the QQ curve lifts well off the diagonal). The random-effects model is **closer to the diagonal but not perfect**: the DerSimonian-Laird estimate of $\tau^2$ is itself noisy with only four studies, so some inflation remains. This is why meta-analyses report heterogeneity (and why variants with a significant $Q$ are inspected by hand).
* In the **forest plot** (right) the squares are the studies (large = precise), the lines are 95 % confidence intervals, the green diamond is the fixed-effect estimate and the pink diamond the random-effects estimate, which is wider.

## 5. Splitting the 20 Butterflies into Two Batches

We build two mini-studies, **B1** and **B2**, each with `N_PER_MORPH_PER_BATCH` *m* and the same number of *i* individuals (5 + 5 = 10). A **stratified** split (done separately within each morph) guarantees that both batches have the same case/control ratio, so that a difference in power between them cannot be caused by an unbalanced design.

**Why does depth matter?** In a real multi-cohort GWAS the studies differ in how they were sequenced. With low-coverage data, a batch with lower depth has more uncertain genotypes, so its test statistics are weaker (and biased towards zero). To *avoid creating a spurious batch effect by chance*, we compare the mean sequencing depth of the batches. `SPLIT_MODE = "balance_depth"` picks, among 2000 random stratified splits, the one with the most similar depth (overall and within each morph). The default is a plain random split.

In [ ]:
N_PER = N_PER_MORPH_PER_BATCH
def stratified_split(rng):
    assign = pd.Series("unused", index=pheno.index, dtype=object)
    for m_ in MORPH_ORDER:
        idx = pheno.index[pheno.morph == m_].to_numpy()
        pick = rng.choice(idx, 2 * N_PER, replace=False)
        assign[pick[:N_PER]] = BATCH_NAMES[0]; assign[pick[N_PER:]] = BATCH_NAMES[1]
    return assign

def depth_imbalance(assign):
    if "mean_depth" not in pheno.columns: return 0.0
    g = lambda mask: pheno.mean_depth[mask].mean()
    s = abs(g(assign == BATCH_NAMES[0]) - g(assign == BATCH_NAMES[1]))
    for m_ in MORPH_ORDER:
        s += abs(g((assign == BATCH_NAMES[0]) & (pheno.morph == m_)) - g((assign == BATCH_NAMES[1]) & (pheno.morph == m_)))
    return s

rng = np.random.default_rng(SPLIT_SEED)
if SPLIT_MODE == "balance_depth" and "mean_depth" in pheno.columns:
    cands = [stratified_split(rng) for _ in range(2000)]
    assign = min(cands, key=depth_imbalance)
    print("balance_depth: chose the best of 2000 random splits")
else:
    assign = stratified_split(rng)
    if SPLIT_MODE == "balance_depth": print("no mean_depth column in phenotypes.tsv: plain random split")
pheno["batch"] = assign
BATCH = {b: pheno[pheno.batch == b].reset_index(drop=True) for b in BATCH_NAMES}
for b in BATCH_NAMES:
    assert len(BATCH[b]) == 2 * N_PER and BATCH[b].y.sum() == N_PER, f"batch {b} is not balanced"
pheno[["sampleID", "morph", "batch"] + (["mean_depth"] if "mean_depth" in pheno.columns else [])].to_csv(D["tables"] / "batch_assignment.tsv", sep="\t", index=False)

summ = pheno[pheno.batch != "unused"].groupby(["batch", "morph"]).agg(n=("sampleID", "size"),
        **({"mean_depth": ("mean_depth", "mean")} if "mean_depth" in pheno.columns else {})).round(2)
display(summ)
if (pheno.batch == "unused").any(): print(f"{(pheno.batch == 'unused').sum()} individuals are not used (more than {2*N_PER} of a morph available).")
for b in BATCH_NAMES:
    print(f"\n{b}: {', '.join(BATCH[b].sampleID)}")

### 5.1 Is the split balanced for sequencing depth?

The left panel shows the mean depth of each individual by batch and morph. The Welch *t*-test compares the two batches: a small p-value would warn us that one batch is systematically deeper, which would later show up as apparent heterogeneity. With 10 individuals per batch the test has little power, so it only catches large differences.

In [ ]:
if "mean_depth" in pheno.columns:
    used = pheno[pheno.batch != "unused"]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    rng_j = np.random.default_rng(0)
    for xi, b in enumerate(BATCH_NAMES):
        for m_ in MORPH_ORDER:
            s = used[(used.batch == b) & (used.morph == m_)]
            ax[0].scatter(xi + (0.12 if m_ == "m" else -0.12) + rng_j.normal(0, 0.03, len(s)), s.mean_depth, color=MORPH_COLORS[m_], s=40, label=f"{m_}" if xi == 0 else None)
    ax[0].set_xticks(range(len(BATCH_NAMES))); ax[0].set_xticklabels(BATCH_NAMES); ax[0].set(ylabel="mean depth (QC notebook)", title="Sequencing depth by batch and morph"); ax[0].legend(title="morph")
    a_, b_ = [used.mean_depth[used.batch == b].values for b in BATCH_NAMES]
    t_, p_ = stats.ttest_ind(a_, b_, equal_var=False)
    ax[1].axis("off")
    ax[1].text(0, 0.9, f"mean depth {BATCH_NAMES[0]}: {a_.mean():.2f}\nmean depth {BATCH_NAMES[1]}: {b_.mean():.2f}\n\nWelch t-test: t = {t_:.2f},  P = {p_:.2f}\n\n"
               + ("No evidence of a depth difference between the batches." if p_ > 0.1 else "Warning: the batches differ in depth; consider SPLIT_MODE = 'balance_depth'."), va="top", fontsize=11)
    plt.tight_layout(); savefig("batch_depth_balance"); plt.show()
else:
    print("phenotypes.tsv has no 'mean_depth' column: skipping the depth balance check.")

### 5.2 Input files for ANGSD (one set per batch)

ANGSD wants plain text files: the **list of BAM files** (one path per line) and the **phenotype** (one number per line, in the *same order* as the BAM list: 1 = *m*, 0 = *i*). The order matters: if the lines do not match, ANGSD silently tests the wrong individuals. We also write the list of sample IDs in the same order so that every ANGSD output can be traced back.

In [ ]:
FILES = {}
for b in BATCH_NAMES:
    df = BATCH[b]
    FILES[b] = dict(bam=D["angsd"] / f"bam_list_{b}.txt", ybin=D["angsd"] / f"pheno_{b}.ybin", ids=D["angsd"] / f"samples_{b}.txt")
    FILES[b]["bam"].write_text("\n".join(pheno.set_index("sampleID").loc[df.sampleID, "bam"]) + "\n")
    FILES[b]["ybin"].write_text("\n".join(map(str, df.y)) + "\n")
    FILES[b]["ids"].write_text("\n".join(df.sampleID) + "\n")
    print(f"{b}: {FILES[b]['bam'].name} ({len(df)} lines);  {FILES[b]['ybin'].name}: {df.y.sum()} cases (1) and {(df.y == 0).sum()} controls (0)")
print("\nFirst three lines of batch B1 side by side (BAM | phenotype):")
for bm, y in zip(open(FILES['B1']['bam']).read().split()[:3], open(FILES['B1']['ybin']).read().split()[:3]):
    print(f"  {bm}  |  {y}")

## 6. Chromosomes and a Depth Pilot for Each Batch

### 6.1 Which chromosomes do we analyse?

We read contig names and lengths from the BAM header, identify the **M chromosome (chromosome 26)** and the **Z chromosome** from the descriptions in the reference FASTA (if available), and analyse the **autosomes only** (females are ZW, so the Z is hemizygous and needs a different model). Every contig gets an integer *index* in header order; later this index is combined with the position into one integer key per variant (Section 7.4).

In [ ]:
hdr = subprocess.run(["samtools", "view", "-H", pheno.bam.iloc[0]], capture_output=True, text=True).stdout
sq = [l.split("\t") for l in hdr.splitlines() if l.startswith("@SQ")]
fai = pd.DataFrame({"contig": [x[1][3:] for x in sq], "length": [int(x[2][3:]) for x in sq]})

ref_candidates = sorted((PROJECT_DIR / "reference_HypMis_v2").glob("*.fa")) if (PROJECT_DIR / "reference_HypMis_v2").exists() else []
if ref_candidates:
    REF_FASTA = ref_candidates[0]
    heads = subprocess.run(["grep", ">", str(REF_FASTA)], capture_output=True, text=True).stdout.splitlines()
    desc = {h[1:].split(" ", 1)[0]: (h[1:].split(" ", 1) + [""])[1] for h in heads}
    fai["description"] = fai.contig.map(desc)
    fai["label"] = fai.description.str.extract(r"(?:chromosome|scaffold)[:\s_]+([A-Za-z0-9]+)", expand=False)
    print("Reference FASTA:", REF_FASTA)
else:
    REF_FASTA = None
    fai["description"] = ""; fai["label"] = np.nan
    print("Reference FASTA not found: chromosome labels are unavailable. Set M_CHROM / Z_CHROM manually below.")

def contig_by_label(label):
    hit = fai.loc[fai.label == str(label), "contig"]
    return hit.iloc[0] if len(hit) else None

M_CHROM = contig_by_label("26")        # set manually if None or wrong, e.g. M_CHROM = "OZ123456.1"
Z_CHROM = contig_by_label("Z")
W_CHROM = contig_by_label("W")

big = fai[fai.length > 5e6]
autosome_mask = big.label.fillna("").str.fullmatch(r"\d+") & ~big.contig.isin([c for c in [Z_CHROM, W_CHROM] if c])
AUTOSOMES = list(big.loc[autosome_mask, "contig"])
if not AUTOSOMES:
    AUTOSOMES = [c for c in big.contig if c != Z_CHROM]
    print("No numeric chromosome labels found: using all sequences > 5 Mb except the Z chromosome.")
ANALYSED = AUTOSOMES + ([Z_CHROM] if (INCLUDE_Z and Z_CHROM) else [])
ANALYSED = [c for c in fai.contig if c in set(ANALYSED)]

fai = fai.set_index("contig", drop=False)
CONTIG_INDEX = {c: i for i, c in enumerate(fai.contig)}          # contig name -> integer index (BAM-header order)
CHR_LABEL = fai.label.fillna("?").to_dict()
REGIONS = D["angsd"] / "regions.txt"
REGIONS.write_text("\n".join(ANALYSED) + "\n")
PLOT_OFFSET, _cum = {}, 0
for c in ANALYSED:
    PLOT_OFFSET[c] = _cum; _cum += int(fai.loc[c, "length"])
PLOT_TOTAL = _cum

print(f"\nSequences in the BAM header : {len(fai)}")
print(f"M chromosome (26)           : {M_CHROM}")
print(f"Z chromosome / W chromosome : {Z_CHROM} / {W_CHROM}")
print(f"Chromosomes analysed        : {len(ANALYSED)} ({fai.loc[ANALYSED, 'length'].sum()/1e6:.1f} Mb of {fai.length.sum()/1e6:.1f} Mb)")
assert M_CHROM is not None, "Chromosome 26 not identified: set M_CHROM manually above (it carries the M locus)."
assert M_CHROM in ANALYSED, "M_CHROM is not in the analysed chromosomes: check the labels."
display(fai.loc[ANALYSED, ["contig", "length", "label"]].reset_index(drop=True).T)

### 6.2 A depth pilot in each batch: choosing `-minInd` and `-setMaxDepth`

Two site-level filters depend on the data of each study, so each batch gets its own pilot (as in a real consortium, where every cohort does its own QC):

* **`-minInd`**: the minimum number of individuals with at least one read at a site. Too low → the allele frequency rests on a handful of individuals; too high → most of the genome is lost at low coverage.
* **`-setMaxDepth`**: the maximum **total** depth over all individuals. Sites with abnormally high depth are typically **collapsed repeats/paralogues** that create false SNPs (the M locus itself is a transposable-element-rich region, so interpret signals there with care).

We measure both on the M chromosome with `-doDepth 1`, and then apply the rules

* `MIN_IND` = half of the expected number of individuals with data at a covered site (bounded between 3 and N/2);
* `MAX_DEPTH` = mean + 3 standard deviations of the total depth.

| Flag | Meaning |
|---|---|
| `-b` | file with the BAM paths |
| `-r chr:` | restrict to one chromosome |
| `-doCounts 1 -doDepth 1` | count bases and write the depth distributions (`.depthSample`, `.depthGlobal`) |
| `-maxDepth 1000` | depths above this are pooled in the last bin |
| `-minMapQ`, `-minQ` | discard reads/bases of low mapping/base quality |
| `-uniqueOnly 1`, `-remove_bads 1` | unique hits only; drop duplicates and QC-failed reads |
| `-only_proper_pairs 0` | our reads are single-end |

In [ ]:
READ_FILTERS = f"-minMapQ {MIN_MAPQ} -minQ {MIN_BASEQ} -uniqueOnly 1 -remove_bads 1 -only_proper_pairs 0"
MAXD_PILOT = 1000
BP = {}                                   # per-batch parameters chosen from the pilot

for b in BATCH_NAMES:
    PILOT = D["angsd"] / f"pilot_depth_{b}"
    if not done(f"{PILOT}.depthSample", f"{PILOT}.depthGlobal"):
        run(f"angsd -b {FILES[b]['bam']} -r {M_CHROM}: -doCounts 1 -doDepth 1 -maxDepth {MAXD_PILOT} {READ_FILTERS} "
            f"-nThreads {THREADS} -out {PILOT}", log=D["angsd"] / f"pilot_depth_{b}.log", show=3)
    ds = np.atleast_2d(np.loadtxt(f"{PILOT}.depthSample")); dg = np.loadtxt(f"{PILOT}.depthGlobal").ravel()
    Nb = len(BATCH[b]); assert ds.shape[0] == Nb, f"depthSample has {ds.shape[0]} rows but batch {b} has {Nb} samples"
    dvals = np.arange(ds.shape[1])
    sites_any = dg[1:].sum(); cov_frac = ds[:, 1:].sum(axis=1) / sites_any
    tot = np.arange(len(dg))[1:]; w = dg[1:]
    mean_tot = (tot * w).sum() / w.sum(); sd_tot = np.sqrt(((tot - mean_tot) ** 2 * w).sum() / w.sum())
    max_auto = int(min(np.ceil(mean_tot + 3 * sd_tot), MAXD_PILOT))
    min_auto = int(np.clip(round(0.5 * cov_frac.sum()), 3, Nb // 2))
    BP[b] = dict(MIN_IND=MIN_IND if MIN_IND is not None else min_auto, MAX_DEPTH=MAX_DEPTH if MAX_DEPTH is not None else max_auto,
                 mean_tot=mean_tot, sd_tot=sd_tot, expected_ind=cov_frac.sum(), dg=dg, cov_frac=cov_frac)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for b in BATCH_NAMES:
    dg = BP[b]["dg"]; shown = min(len(dg), int(BP[b]["mean_tot"] + 6 * BP[b]["sd_tot"]) + 10)
    ax[0].plot(np.arange(1, shown), dg[1:shown], color=BATCH_COLORS[b], label=f"{b}: -setMaxDepth {BP[b]['MAX_DEPTH']}")
    ax[0].axvline(BP[b]["MAX_DEPTH"], color=BATCH_COLORS[b], ls="--")
    ax[1].hist(BP[b]["cov_frac"], bins=10, alpha=0.6, color=BATCH_COLORS[b], label=b)
ax[0].set(yscale="log", xlabel="total depth at a site (all individuals of the batch)", ylabel="number of sites", title="Global depth distribution (M chromosome)"); ax[0].legend()
ax[1].set(xlabel="fraction of covered sites at which the individual has data", ylabel="individuals", title="Per-individual data availability"); ax[1].legend()
plt.tight_layout(); savefig("depth_pilot"); plt.show()
print(pd.DataFrame({b: {"expected individuals with data": round(BP[b]["expected_ind"], 1), "mean total depth": round(BP[b]["mean_tot"], 1),
                        "-minInd": BP[b]["MIN_IND"], "-setMaxDepth": BP[b]["MAX_DEPTH"]} for b in BATCH_NAMES}))

## 7. Variant Calling and Association Testing with ANGSD, Separately in Each Batch

### 7.1 Probabilistic variant calling (genotype likelihoods)

At low coverage a genotype cannot be called reliably from a handful of reads, so ANGSD does not call genotypes: it computes, at every site and for every individual, the **genotype likelihood** $P(\text{reads}\mid g)$ for the three genotypes $g\in\{AA, Aa, aa\}$, estimates the allele frequency from all individuals, and decides whether the site is polymorphic with a likelihood-ratio test (`-SNP_pval`). Association is then tested **on the likelihoods/posteriors**, so that genotype uncertainty is propagated into the test.

We run ANGSD **twice per batch**: once to write the genotype likelihoods and allele frequencies (`.beagle.gz`, `.mafs.gz`; we need them in Section 7.5), and once with `-doAsso` for the association test. Both runs use the same filters, so they see the same SNPs.

| Flag | Meaning |
|---|---|
| `-b list` | BAM list of the batch (10 individuals) |
| `-rf regions.txt` | restrict to the autosomes |
| `-GL 2` | GATK genotype-likelihood model |
| `-doMajorMinor 1` | infer the major and minor allele from the genotype likelihoods **of this batch** |
| `-doMaf 1` | estimate the minor-allele frequency by EM |
| `-doCounts 1` | count bases (needed by the depth filters) |
| `-minInd`, `-setMaxDepth` | site filters from the batch's depth pilot (Section 6.2) |
| `-SNP_pval 1e-6` | call a SNP if the likelihood-ratio test of monomorphism has P < 1e-6 |
| `-minMaf` | minimum allele frequency |
| `-doPost 1` | genotype posteriors with the estimated allele frequency as prior (HWE prior) |
| `-doGlf 2` | write the genotype likelihoods in BEAGLE format |

> **Why the major/minor allele can differ between batches.** `-doMajorMinor 1` picks the two most likely alleles *in each batch separately*. At a site with allele frequency close to 0.5 the "minor" allele in batch B1 may be the "major" allele in B2. The effect allele of the association test is the **minor** allele *of that batch*. This is exactly the situation in which a naive meta-analysis would **cancel a real effect**; Section 8 handles it explicitly.

In [ ]:
CALL, ASSO = {}, {}
for b in BATCH_NAMES:
    CALL[b] = D["angsd"] / f"calls_{b}"
    common = (f"-b {FILES[b]['bam']} -rf {REGIONS} -GL {GL_MODEL} -doMajorMinor 1 -doMaf 1 -doCounts 1 {READ_FILTERS} "
              f"-minInd {BP[b]['MIN_IND']} -setMaxDepth {BP[b]['MAX_DEPTH']} -SNP_pval {SNP_PVAL} -minMaf {MIN_MAF} -nThreads {THREADS}")
    if not done(f"{CALL[b]}.beagle.gz", f"{CALL[b]}.mafs.gz"):
        run(f"angsd {common} -doPost 1 -doGlf 2 -out {CALL[b]}", log=D["angsd"] / f"angsd_calls_{b}.log", show=4)
    else:
        print(f"[{b}] ANGSD calls already exist: skipping the run.")
    assert Path(f"{CALL[b]}.beagle.gz").exists() and Path(f"{CALL[b]}.mafs.gz").exists(), (
        f"ANGSD did not produce the expected files for {b}: see {D['angsd'] / ('angsd_calls_' + b + '.log')}. "
        "Typical causes: no site passed the filters (lower MIN_IND or MIN_MAF), or a BAM could not be read.")
    n_snp = int(run(f"zcat {CALL[b]}.mafs.gz | tail -n +2 | wc -l", quiet=True).stdout.strip() or 0)
    print(f"[{b}] SNPs called by ANGSD: {n_snp:,}")
    assert n_snp > 0, f"No SNP was called in batch {b}: lower MIN_IND / MIN_MAF or raise SNP_PVAL (Section 3.1)."

### 7.2 The output files: `.mafs.gz` and `.beagle.gz`

* **`.mafs.gz`**: one row per SNP: chromosome, position, major and minor allele, reference allele, the estimated minor-allele frequency (`knownEM`), and the number of individuals with data (`nInd`).
* **`.beagle.gz`**: one row per SNP: `marker` (`chromosome_position`), allele 1 (major), allele 2 (minor), then **three genotype likelihoods per individual** (AA, Aa, aa), scaled to sum to 1. A site where an individual has no reads shows the flat triple 0.333, 0.333, 0.333.

Alleles are coded 0 = A, 1 = C, 2 = G, 3 = T in the BEAGLE file.

In [ ]:
b = BATCH_NAMES[0]
mafs_head = pd.read_csv(f"{CALL[b]}.mafs.gz", sep="\t", nrows=5)
mafs_head = mafs_head.loc[:, ~mafs_head.columns.str.startswith("Unnamed")]
print(f"--- first rows of the .mafs file of {b} ---")
display(mafs_head)
print(f"--- first rows of the .beagle file of {b} (first 2 of {len(BATCH[b])} individuals shown) ---")
with gzip.open(f"{CALL[b]}.beagle.gz", "rt") as fh:
    rows = [fh.readline().rstrip("\n").split("\t") for _ in range(4)]
display(pd.DataFrame(rows[1:], columns=rows[0])[rows[0][:9]])

### 7.3 Association testing with `-doAsso`

ANGSD can test association inside the same program that computes the genotype likelihoods, using the binary phenotype (`-yBin`: 1 = *m*, 0 = *i*). We use **`-doAsso 2`**, the **score test**: it models the log-odds of being an *m* individual as a linear function of the *expected genotype* (the "dosage" $E[g\mid\text{reads}]$, from `-doPost 1`). Its statistic `LRT` follows a $\chi^2_1$ distribution under the null; the p-value is $P=1-F_{\chi^2_1}(\mathrm{LRT})$.

The ANGSD score test returns **only the test statistic**, not an effect size or a standard error. But a meta-analysis by **inverse variance** needs $\hat\beta$ and $\mathrm{SE}$. Section 7.5 shows how to obtain them from the *same* dosages in a way that is exactly consistent with the score test.

| Flag | Meaning |
|---|---|
| `-yBin pheno.ybin` | binary phenotype, one value per line in BAM-list order |
| `-doAsso 2` | score test |
| `-doPost 1` | genotype posteriors with the allele-frequency prior |
| `-minHigh h` | require at least $h$ individuals with a posterior $>0.9$ in two genotype classes; far too strict for low-coverage data → 0 (off) |
| `-minCount c` | require an expected minor-allele count $\ge c$ (frequency × 2N). With 10 individuals, `c = 5` means MAF ≥ 0.25 |

> **Caveat.** With 10 individuals per batch the number of testable SNPs is limited by `-minCount` and the p-values are approximate: small samples inflate or deflate score-test statistics. We therefore check $\lambda_{GC}$ and the QQ plots.

The result file has **one row per site, including sites where the test failed** (statistic −999). "Number of sites in the file" and "number of sites tested" are different numbers.

In [ ]:
for b in BATCH_NAMES:
    ASSO[b] = D["angsd"] / f"assoc_{b}"
    common = (f"-b {FILES[b]['bam']} -rf {REGIONS} -GL {GL_MODEL} -doMajorMinor 1 -doMaf 1 -doCounts 1 {READ_FILTERS} "
              f"-minInd {BP[b]['MIN_IND']} -setMaxDepth {BP[b]['MAX_DEPTH']} -SNP_pval {SNP_PVAL} -minMaf {MIN_MAF} -nThreads {THREADS}")
    if not done(f"{ASSO[b]}.lrt0.gz") and not glob.glob(f"{ASSO[b]}.lrt*.gz"):
        run(f"angsd {common} -doPost 1 -yBin {FILES[b]['ybin']} -doAsso {ASSO_MODE} -minHigh {ASSO_MIN_HIGH} -minCount {ASSO_MIN_COUNT} -out {ASSO[b]}",
            log=D["angsd"] / f"angsd_assoc_{b}.log", show=4)
    else:
        print(f"[{b}] ANGSD -doAsso {ASSO_MODE}: output exists, skipping.")
    f = sorted(glob.glob(f"{ASSO[b]}.lrt*.gz"))
    assert f, f"No association output for {b}: see {D['angsd'] / ('angsd_assoc_' + b + '.log')}"
    print(f"[{b}] result file: {Path(f[0]).name}  ({Path(f[0]).stat().st_size/1e6:.1f} MB)")
print("\nFirst rows of the association result of", BATCH_NAMES[0], ":")
display(pd.read_csv(sorted(glob.glob(f"{ASSO[BATCH_NAMES[0]]}.lrt*.gz"))[0], sep="\t", nrows=5))

### 7.4 Reading the results: one integer key per variant

A whole-genome file can contain millions of variants. Python strings are expensive (each `"OZ123456.1_6731020"` takes ~70 bytes), and merging two such tables by string can exhaust Colab's 12 GB of memory. We therefore represent every variant by **one integer**:

$$\text{key}=\text{contig index}\times10^{9}+\text{position},$$

where the contig index is the position of the contig in the BAM header (Section 6.1). Keys are numerical, sorted by genome position, and can be matched between batches with `numpy` in a fraction of a second. The same integer is also used as the *marker name* in the GWAMA and METAL input files (for example `26006731020` = contig 26, position 6,731,020), and translated back when reading their outputs.

### 7.5 Effect sizes and standard errors from the genotype dosages

For each individual $j$ with phenotype $y_j\in\{0,1\}$ and expected genotype ("dosage") $x_j=E[g_j\mid\text{reads}]\in[0,2]$ (copies of the minor allele, computed from the genotype likelihoods and the HWE prior with the ANGSD allele frequency), the **score statistic** of the logistic model $\mathrm{logit}\,P(y=1)=\alpha+\beta x$ at $\beta=0$ is

$$U=\sum_j (y_j-\bar y)\,x_j,\qquad I=\bar y(1-\bar y)\sum_j (x_j-\bar x)^2,\qquad Z=\frac{U}{\sqrt I},\qquad \chi^2_{score}=Z^2\approx\mathrm{LRT}.$$

The same quantities give a **one-step estimate of the effect size and its standard error**:

$$\hat\beta=\frac{U}{I}\ \ (\text{log odds ratio per minor allele}),\qquad \mathrm{SE}=\frac{1}{\sqrt I},\qquad\text{so that}\ \ \hat\beta/\mathrm{SE}=Z.$$

* The **direction** of the effect is the sign of $U$: positive means the minor allele (more precisely, the dosage) is higher in the *m*-morphs than in the *i*-morphs.
* We keep **ANGSD's own LRT** as the measure of significance: $Z=\mathrm{sign}(U)\sqrt{\mathrm{LRT}}$ (`EFFECT_SOURCE = "angsd_lrt"`), and take the standard error from the information $I$, so that $\hat\beta=Z\cdot\mathrm{SE}$. Alternatively `EFFECT_SOURCE = "score"` uses our own $Z$ throughout. Section 7.6 compares the two to show that they agree.
* **Interpretation caveat.** $\hat\beta$ is a *score-based approximation* of the log odds ratio. It is accurate for modest effects, but is not the maximum-likelihood estimate; for very strong effects (such as the M locus, which separates the morphs almost perfectly) the true log odds ratio is larger. For meta-analysis, what matters is that both batches use the **same scale**, which they do.

In [ ]:
ALLELE_CODE = {"A": 0, "C": 1, "G": 2, "T": 3}
ALLELE_LETTER = np.array(list("ACGTN"))

def read_assoc_lean(path):
    # Read an ANGSD .lrt0.gz file into compact numpy arrays (chromosome -> integer key).
    hdr = pd.read_csv(path, sep="\t", nrows=0).columns
    lrt_col = [c for c in hdr if str(c).upper().startswith("LRT")][0]
    want = ["Chromosome", "Position", "Major", "Minor", "Frequency", lrt_col]
    miss = [c for c in want if c not in hdr]
    assert not miss, f"unexpected columns in {path}: missing {miss}; found {list(hdr)}"
    parts = {k: [] for k in ["key", "major", "minor", "freq", "lrt"]}
    for ch in pd.read_csv(path, sep="\t", usecols=want, chunksize=CHUNK, dtype={"Chromosome": str, "Major": str, "Minor": str}):
        ci = ch["Chromosome"].map(CONTIG_INDEX)
        good = ci.notna().to_numpy(); ch = ch[good]; ci = ci[good].to_numpy(np.int64)
        parts["key"].append(ci * KEYMULT + ch["Position"].to_numpy(np.int64))
        parts["major"].append(ch["Major"].map(ALLELE_CODE).fillna(4).to_numpy(np.int8))
        parts["minor"].append(ch["Minor"].map(ALLELE_CODE).fillna(4).to_numpy(np.int8))
        parts["freq"].append(pd.to_numeric(ch["Frequency"], errors="coerce").to_numpy(np.float32))
        parts["lrt"].append(pd.to_numeric(ch[lrt_col], errors="coerce").to_numpy(np.float32))
    A = {k: np.concatenate(v) for k, v in parts.items()}
    o = np.argsort(A["key"], kind="stable")
    return {k: v[o] for k, v in A.items()}

def dosage_stats(beagle, mafs, y):
    # One pass over the BEAGLE genotype likelihoods: dosage-based score quantities U and I (Section 7.5).
    y = np.asarray(y, dtype=np.float64); ybar = y.mean(); case = y == 1
    parts = {k: [] for k in ["key", "major", "minor", "sign", "se", "z", "f_case", "f_ctrl", "n_inf"]}
    rb = pd.read_csv(beagle, sep="\t", chunksize=CHUNK)
    rm = pd.read_csv(mafs, sep="\t", chunksize=CHUNK, usecols=["chromo", "position", "major", "minor", "knownEM"],
                     dtype={"chromo": str, "major": str, "minor": str})
    for cb, cm in zip(rb, rm):
        m = len(cb); assert len(cm) == m, "BEAGLE and MAFS files have a different number of rows: they must come from the same ANGSD run"
        n = (cb.shape[1] - 3) // 3; assert n == len(y), f"BEAGLE has {n} individuals but the phenotype has {len(y)}"
        G = cb.iloc[:, 3:].to_numpy(np.float64).reshape(m, n, 3)
        flat = (G.max(-1) - G.min(-1)) < 1e-3                             # flat likelihood = no reads for this individual
        G = G / np.maximum(G.sum(-1, keepdims=True), 1e-300)
        p = np.clip(np.nan_to_num(cm["knownEM"].to_numpy(np.float64), nan=0.5), 1e-6, 1 - 1e-6)
        prior = np.stack([(1 - p) ** 2, 2 * p * (1 - p), p ** 2], 1)      # Hardy-Weinberg prior for 0, 1, 2 minor alleles
        post = G * prior[:, None, :]; post /= post.sum(-1, keepdims=True) + 1e-300
        x = post[..., 1] + 2 * post[..., 2]                               # dosage = expected number of minor alleles
        xc = x - x.mean(1, keepdims=True)
        Sxx = (xc ** 2).sum(1)
        U = (xc * (y - ybar)).sum(1)
        info = ybar * (1 - ybar) * Sxx
        with np.errstate(divide="ignore", invalid="ignore"):
            se = 1.0 / np.sqrt(info); z = U / np.sqrt(info)
        ci = cm["chromo"].map(CONTIG_INDEX); good = ci.notna().to_numpy()
        parts["key"].append(np.where(good, ci.fillna(0).to_numpy(np.int64) * KEYMULT + cm["position"].to_numpy(np.int64), -1))
        parts["major"].append(cm["major"].map(ALLELE_CODE).fillna(4).to_numpy(np.int8))
        parts["minor"].append(cm["minor"].map(ALLELE_CODE).fillna(4).to_numpy(np.int8))
        parts["sign"].append(np.sign(U).astype(np.int8)); parts["se"].append(se.astype(np.float32)); parts["z"].append(z.astype(np.float32))
        parts["f_case"].append((x[:, case].mean(1) / 2).astype(np.float32)); parts["f_ctrl"].append((x[:, ~case].mean(1) / 2).astype(np.float32))
        parts["n_inf"].append((~flat).sum(1).astype(np.int8))
    R = {k: np.concatenate(v) for k, v in parts.items()}
    keep = R["key"] >= 0
    R = {k: v[keep] for k, v in R.items()}
    o = np.argsort(R["key"], kind="stable")
    return {k: v[o] for k, v in R.items()}

def build_batch(b):
    # Association result + dosage statistics -> one compact summary-statistics table (numpy arrays) for batch b.
    cache = D["summary_stats"] / f"stats_{b}.npz"
    if done(cache):
        z = np.load(cache); print(f"[{b}] summary statistics loaded from cache: {cache.name}")
        return {k: z[k] for k in z.files}
    A = read_assoc_lean(sorted(glob.glob(f"{ASSO[b]}.lrt*.gz"))[0])
    Dd = dosage_stats(f"{CALL[b]}.beagle.gz", f"{CALL[b]}.mafs.gz", BATCH[b].y.to_numpy(float))
    assert len(np.unique(A["key"])) == len(A["key"]), "duplicated positions in the ANGSD association file"
    idx = np.minimum(np.searchsorted(Dd["key"], A["key"]), len(Dd["key"]) - 1)
    found = Dd["key"][idx] == A["key"]
    same = found & (Dd["major"][idx] == A["major"]) & (Dd["minor"][idx] == A["minor"])
    n = len(A["key"])
    S = dict(key=A["key"], major=A["major"], minor=A["minor"], freq=A["freq"], lrt=A["lrt"],
             tested=(np.isfinite(A["lrt"]) & (A["lrt"] > -1)),
             sign=np.where(same, Dd["sign"][idx], 0).astype(np.int8),
             se=np.where(same, Dd["se"][idx], np.nan).astype(np.float32),
             z_score=np.where(same, Dd["z"][idx], np.nan).astype(np.float32),
             f_case=np.where(same, Dd["f_case"][idx], np.nan).astype(np.float32),
             f_ctrl=np.where(same, Dd["f_ctrl"][idx], np.nan).astype(np.float32),
             n_inf=np.where(same, Dd["n_inf"][idx], 0).astype(np.int8))
    S["n_not_in_beagle"] = np.array([int((~found).sum())]); S["n_allele_mismatch"] = np.array([int((found & ~same).sum())])
    np.savez_compressed(cache, **S)
    return S

STAT = {b: build_batch(b) for b in BATCH_NAMES}
for b in BATCH_NAMES:
    S = STAT[b]
    print(f"[{b}] sites in the association file: {len(S['key']):,}; tested: {int(S['tested'].sum()):,}; "
          f"not found in the BEAGLE file: {int(S['n_not_in_beagle'][0]):,}; allele mismatch between the two ANGSD runs: {int(S['n_allele_mismatch'][0]):,}")

### 7.6 From statistics to meta-analysis input: signed *Z*, β, SE, P, and a check against ANGSD

For each batch and each tested SNP we now define

* `z` = the signed *Z*-score ($\mathrm{sign}(U)\sqrt{\mathrm{LRT}}$, or our own score $Z$ if `EFFECT_SOURCE = "score"`);
* `beta` = $Z\cdot\mathrm{SE}$, `se` from the information $I$;
* `p` = $2\Phi(-|Z|)$ (identical to the $\chi^2_1$ p-value of the LRT).

**Check:** our own score statistic, recomputed from the genotype likelihoods, should reproduce ANGSD's LRT closely (correlation of the signed *Z* near 1, signs identical). Differences are expected because ANGSD's EM and our direct calculation treat missing data and the prior slightly differently. If the agreement were poor, something would be wrong with the file matching or the phenotype order.

In [ ]:
def finalize(S, source=EFFECT_SOURCE):
    lrt = np.where(S["tested"], S["lrt"], np.nan).astype(np.float64)
    z_angsd = S["sign"].astype(np.float64) * np.sqrt(np.clip(lrt, 0, None))
    z = z_angsd if source == "angsd_lrt" else S["z_score"].astype(np.float64)
    ok = S["tested"] & np.isfinite(z) & np.isfinite(S["se"]) & (S["se"] > 0)
    if source == "angsd_lrt": ok &= (S["sign"] != 0)               # sign 0 = no dosage statistics for this site
    z = np.where(ok, z, np.nan)
    S["z_angsd"] = z_angsd.astype(np.float32)
    S["z"] = z.astype(np.float32)
    S["beta"] = (z * S["se"]).astype(np.float32)
    S["p"] = 2 * stats.norm.sf(np.abs(z))                     # float64: p-values can be very small
    S["ok"] = ok
    S["ci"] = (S["key"] // KEYMULT).astype(np.int32); S["pos"] = (S["key"] % KEYMULT).astype(np.int64)
    return S

for b in BATCH_NAMES: finalize(STAT[b])

fig, ax = plt.subplots(1, len(BATCH_NAMES), figsize=(10, 4.2))
rows = []
for a, b in zip(ax, BATCH_NAMES):
    S = STAT[b]; m = S["ok"] & np.isfinite(S["z_angsd"]) & np.isfinite(S["z_score"])
    r = np.corrcoef(S["z_angsd"][m], S["z_score"][m])[0, 1] if m.sum() > 2 else np.nan
    sgn = (np.sign(S["z_angsd"][m]) == np.sign(S["z_score"][m])).mean() if m.sum() else np.nan
    sub = np.random.default_rng(0).choice(np.where(m)[0], min(m.sum(), 50000), replace=False) if m.sum() else []
    a.scatter(S["z_score"][sub], S["z_angsd"][sub], s=3, alpha=0.3, color=BATCH_COLORS[b], rasterized=True)
    lim = np.nanmax(np.abs(S["z_angsd"][m])) * 1.05 if m.sum() else 1
    a.plot([-lim, lim], [-lim, lim], "k--", lw=1); a.set(xlabel="Z recomputed from dosages (our score statistic)", ylabel="signed Z from ANGSD LRT", title=f"{b}: r = {r:.3f}")
    rows.append({"batch": b, "SNPs compared": int(m.sum()), "Pearson r of Z": r, "same sign": sgn, "median ANGSD Z^2 / our Z^2": float(np.nanmedian(S["z_angsd"][m] ** 2 / np.maximum(S["z_score"][m] ** 2, 1e-12))) if m.sum() else np.nan})
plt.tight_layout(); savefig("check_angsd_vs_score"); plt.show()
display(pd.DataFrame(rows).set_index("batch").round(3))
assert all(STAT[b]["ok"].sum() > 0 for b in BATCH_NAMES), "no SNP was tested in at least one batch: lower ASSO_MIN_COUNT (Section 3.1) and re-run Section 7.3"

### 7.7 The two single-batch results: Manhattan plots side by side

A **Manhattan plot** shows $-\log_{10}P$ for every tested SNP along the genome (chromosomes alternate in colour; the labels are the chromosome numbers). The **red dashed line** is the Bonferroni threshold for the number of SNPs tested in that batch ($0.05/n_{tests}$); the **red points** are SNPs in the M-locus window on chromosome 26. Both panels use **the same vertical scale**, so you can compare the strength of the signals by eye.

To keep the figure light, all SNPs with $-\log_{10}P>2$ are drawn, but only a random subset of the rest.

In [ ]:
OFF_ARR = np.full(len(fai), np.nan); PAR_ARR = np.zeros(len(fai), dtype=int)
for i, c in enumerate(ANALYSED):
    OFF_ARR[CONTIG_INDEX[c]] = PLOT_OFFSET[c]; PAR_ARR[CONTIG_INDEX[c]] = i % 2
M_IDX = CONTIG_INDEX[M_CHROM]
CENTERS = [PLOT_OFFSET[c] + fai.loc[c, "length"] / 2 for c in ANALYSED]

def in_peak_arr(ci, pos):
    return (ci == M_IDX) & (pos >= PEAK_START) & (pos <= PEAK_END)

def manhattan_arrays(ax, ci, pos, p, title, color, n_tests=None, ymax=None, seed=0, thr_label=True):
    sel = np.isfinite(p) & np.isfinite(OFF_ARR[ci]); ci, pos, p = ci[sel], pos[sel], p[sel]
    logp = -np.log10(np.clip(p, 1e-300, 1.0))
    keep = logp > 2
    rest = np.where(~keep)[0]
    if len(rest) > 150_000: keep[np.random.default_rng(seed).choice(rest, 150_000, replace=False)] = True
    else: keep[rest] = True
    x = OFF_ARR[ci[keep]] + pos[keep]; y = logp[keep]; par = PAR_ARR[ci[keep]]
    ax.scatter(x[par == 0], y[par == 0], s=4, color=color, alpha=0.8, linewidths=0, rasterized=True)
    ax.scatter(x[par == 1], y[par == 1], s=4, color="#7f7f7f", alpha=0.8, linewidths=0, rasterized=True)
    pk = in_peak_arr(ci, pos)
    if pk.any():
        ax.scatter(OFF_ARR[ci[pk]] + pos[pk], logp[pk], s=22, color="red", zorder=5, label="M locus window")
        ax.legend(loc="upper left", fontsize=8)
    if n_tests:
        ax.axhline(-np.log10(0.05 / n_tests), c="r", ls="--", lw=1)
    ax.set_xticks(CENTERS); ax.set_xticklabels([CHR_LABEL[c] for c in ANALYSED], fontsize=7)
    ax.set_xlim(0, PLOT_TOTAL)
    if ymax: ax.set_ylim(0, ymax)
    ax.set(ylabel=r"$-\log_{10}$ P", xlabel="chromosome", title=title)

def qq_arrays(ax, p, color, label=None, max_points=8000):
    p = np.sort(np.asarray(p, dtype=np.float64)); p = p[np.isfinite(p) & (p > 0)]
    n = len(p)
    if n == 0: return
    idx = np.unique(np.round(np.geomspace(1, n, max_points)).astype(np.int64)) - 1      # all scales, dense in the tail
    exp = -np.log10((idx + 0.5) / n); obs = -np.log10(p[idx])
    lam = lambda_gc(p)
    ax.scatter(exp, obs, s=7, color=color, label=(f"{label}  λ={lam:.2f}" if label else f"λ={lam:.2f}"))
    m = max(exp.max(), obs.max()) * 1.02
    ax.plot([0, m], [0, m], "k--", lw=1)
    ax.set(xlabel="expected -log10 P", ylabel="observed -log10 P"); ax.legend(fontsize=8)

n_tests = {b: int(STAT[b]["ok"].sum()) for b in BATCH_NAMES}
ymax = 1.08 * max(np.nanmax(-np.log10(np.clip(STAT[b]["p"][STAT[b]["ok"]], 1e-300, 1))) for b in BATCH_NAMES)
ymax = max(ymax, max(-np.log10(0.05 / n) for n in n_tests.values()) * 1.05)
fig, ax = plt.subplots(1, 2, figsize=(22, 5.2), sharey=True)
for a, b in zip(ax, BATCH_NAMES):
    S = STAT[b]
    manhattan_arrays(a, S["ci"][S["ok"]], S["pos"][S["ok"]], S["p"][S["ok"]], f"{b}: ANGSD -doAsso {ASSO_MODE}, {len(BATCH[b])} individuals, {n_tests[b]:,} SNPs tested", BATCH_COLORS[b], n_tests=n_tests[b], ymax=ymax)
plt.tight_layout(); savefig("manhattan_two_batches"); plt.show()

**How to read the pair of Manhattan plots.** If a real effect exists, it should appear at the **same genomic position in both batches**, even when neither batch reaches the Bonferroni line on its own (10 individuals have little power). A peak that is present in one batch only is a candidate false positive (or a batch-specific artefact such as a collapsed repeat that attracts reads in one library only). In Section 8 we combine the two.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
rows = []
for a, b in zip(ax, BATCH_NAMES):
    S = STAT[b]; p = S["p"][S["ok"]]
    qq_arrays(a, p, BATCH_COLORS[b], label=b); a.set_title(f"QQ plot, {b}")
    rows.append({"batch": b, "individuals": len(BATCH[b]), "SNPs tested": len(p), "min P": p.min(), "lambda_GC": lambda_gc(p),
                 "SNPs with P < 1e-4": int((p < 1e-4).sum()), "Bonferroni threshold": 0.05 / len(p),
                 "SNPs below Bonferroni": int((p < 0.05 / len(p)).sum())})
display(pd.DataFrame(rows).set_index("batch").style.format({"min P": "{:.2e}", "lambda_GC": "{:.3f}", "Bonferroni threshold": "{:.2e}"}))
plt.tight_layout(); savefig("qq_two_batches"); plt.show()

**Interpreting λ.** $\lambda_{GC}$ is the median observed $\chi^2$ divided by its expectation under the null (0.455). $\lambda\approx1$: well calibrated; $\lambda>1$: inflation (population structure, relatedness, genotype uncertainty, too small expected counts); $\lambda<1$: deflation (over-conservative tests, common with very small samples and strict `-minCount`). Both are reasons to treat the p-values of a 10-individual batch with caution. The meta-analysis inherits the calibration of its inputs. Note that a QQ plot of a study that contains a **true large-effect locus** (here the M locus) can show a few extreme points without inflation of the bulk.

## 8. Preparing the Meta-Analysis: Overlap, Allele Alignment and Quality Control

Only variants tested in **both** batches can be meta-analysed (a variant seen in one study only is simply reported from that study). We now

1. find the variants tested in both batches (intersection of the integer keys),
2. check whether their alleles match, are swapped (→ flip the effect of batch 2) or are incompatible (→ drop),
3. check that the two batches look like samples of the same population (allele frequencies should correlate strongly) and look at how the effect estimates relate to each other,
4. assemble the **meta-analysis input table**.

### 8.1 Overlap of the variant sets and allele harmonisation

Let batch B1 define the **effect allele** (EA = its minor allele) and the **other allele** (NEA = its major allele). For batch B2:

| Alleles of B2 compared with B1 | Action |
|---|---|
| minor₂ = minor₁ and major₂ = major₁ | **same** orientation: keep |
| minor₂ = major₁ and major₂ = minor₁ | **swapped**: flip B2: $\hat\beta_2\to-\hat\beta_2$, $f_2\to1-f_2$ |
| anything else | **incompatible** (e.g. a third allele): drop the variant |

In [ ]:
S1, S2 = STAT[BATCH_NAMES[0]], STAT[BATCH_NAMES[1]]
k1 = np.where(S1["ok"])[0]; k2 = np.where(S2["ok"])[0]
common, i1, i2 = np.intersect1d(S1["key"][k1], S2["key"][k2], assume_unique=True, return_indices=True)
i1 = k1[i1]; i2 = k2[i2]
same = (S2["minor"][i2] == S1["minor"][i1]) & (S2["major"][i2] == S1["major"][i1])
flip = (S2["minor"][i2] == S1["major"][i1]) & (S2["major"][i2] == S1["minor"][i1])
good = same | flip

n1, n2, nc = len(k1), len(k2), len(common)
ov = pd.DataFrame({"count": [n1 - nc, n2 - nc, nc, int(same.sum()), int(flip.sum()), int((~good).sum()), int(good.sum())]},
                  index=[f"tested in {BATCH_NAMES[0]} only", f"tested in {BATCH_NAMES[1]} only", "tested in both",
                         "  of which: same allele orientation", "  of which: alleles swapped (flip B2)", "  of which: incompatible alleles (dropped)", "variants entering the meta-analysis"])
ov["% of 'tested in both'"] = [np.nan, np.nan] + list((100 * ov["count"].iloc[2:] / max(nc, 1)).round(1))
display(ov)
assert good.sum() > 0, "No SNP is shared by both batches with compatible alleles: relax MIN_MAF / ASSO_MIN_COUNT (Section 3.1) and re-run Sections 7.1-7.6."

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(["only " + BATCH_NAMES[0], "both", "only " + BATCH_NAMES[1]], [n1 - nc, nc, n2 - nc], color=[BATCH_COLORS[BATCH_NAMES[0]], "#009E73", BATCH_COLORS[BATCH_NAMES[1]]])
ax[0].set(ylabel="tested SNPs", title="Variants tested per batch")
ax[1].bar(["same", "swapped", "incompatible"], [int(same.sum()), int(flip.sum()), int((~good).sum())], color=["#009E73", "#E69F00", "#D55E00"])
ax[1].set(ylabel="shared SNPs", title="Allele orientation of B2 relative to B1")
plt.tight_layout(); savefig("overlap_alleles"); plt.show()

### 8.2 Do the batches look like the same population? Frequencies and effects

* **Effect-allele frequency, B1 vs B2** (after flipping): for two samples of the same population the points lie on the diagonal with moderate scatter (sampling noise of 20 chromosomes: SD ≈ $\sqrt{p(1-p)/20}\approx0.11$ at $p=0.5$). A strong departure would mean that the studies differ in ancestry or in a technical artefact, and a meta-analysis would be inappropriate.
* **Z-scores, B1 vs B2.** Under the null there is no correlation. A positive correlation (especially among the top SNPs) means that the same variants and the same directions are found in both batches, i.e. replication.

In [ ]:
g = np.where(good)[0]
j1, j2 = i1[g], i2[g]; fl = flip[g]
f1 = S1["freq"][j1].astype(np.float64); f2 = np.where(fl, 1 - S2["freq"][j2], S2["freq"][j2]).astype(np.float64)
z1 = S1["z"][j1].astype(np.float64); z2 = np.where(fl, -S2["z"][j2], S2["z"][j2]).astype(np.float64)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
sub = np.random.default_rng(1).choice(len(g), min(len(g), 60000), replace=False)
ax[0].scatter(f1[sub], f2[sub], s=3, alpha=0.25, color="grey", rasterized=True)
ax[0].scatter(f1[sub][fl[sub]], f2[sub][fl[sub]], s=5, alpha=0.5, color="#E69F00", label="B2 flipped", rasterized=True)
ax[0].plot([0, 1], [0, 1], "k--", lw=1)
ax[0].set(xlabel=f"effect-allele frequency, {BATCH_NAMES[0]}", ylabel=f"effect-allele frequency, {BATCH_NAMES[1]} (harmonised)", title=f"Allele frequencies: r = {np.corrcoef(f1, f2)[0, 1]:.3f}"); ax[0].legend()
ax[1].scatter(z1[sub], z2[sub], s=3, alpha=0.25, color="grey", rasterized=True)
big_ = (np.abs(z1) > 3) | (np.abs(z2) > 3)
ax[1].scatter(z1[big_], z2[big_], s=10, color="#D55E00", label="|Z| > 3 in either batch")
ax[1].axhline(0, c="k", lw=.6); ax[1].axvline(0, c="k", lw=.6)
ax[1].set(xlabel=f"Z, {BATCH_NAMES[0]}", ylabel=f"Z, {BATCH_NAMES[1]} (harmonised)", title=f"Z-scores: Pearson r = {np.corrcoef(z1, z2)[0, 1]:.3f},  Spearman = {stats.spearmanr(z1[sub], z2[sub])[0]:.3f}"); ax[1].legend()
plt.tight_layout(); savefig("qc_batches_freq_z"); plt.show()
print(f"Fraction of shared SNPs with the same sign of Z in both batches: {(np.sign(z1) == np.sign(z2)).mean():.3f}  (0.5 expected if nothing is shared)")

### 8.3 Assembling the meta-analysis input

The next cell builds one compact table (one row per variant entering the meta-analysis) with, for each batch, the **harmonised** effect size, standard error, *Z*, p-value, effect-allele frequency and number of individuals with data. Optionally (`APPLY_GC = True`) each batch's standard errors are multiplied by $\sqrt{\lambda_{GC}}$ when its $\lambda_{GC}>1$ (genomic control, Section 0.9).

Effect allele (EA) and other allele (NEA) are those of batch 1.

In [ ]:
lam = {b: lambda_gc(STAT[b]["p"][STAT[b]["ok"]]) for b in BATCH_NAMES}
gc_fac = {b: (np.sqrt(max(lam[b], 1.0)) if APPLY_GC else 1.0) for b in BATCH_NAMES}
print("genomic-control factors applied to the standard errors:", {b: round(v, 3) for b, v in gc_fac.items()}, "(APPLY_GC =", APPLY_GC, ")")

sgn2 = np.where(fl, -1.0, 1.0)
b1 = S1["beta"][j1].astype(np.float64)                                   # beta unchanged; only the SE is inflated by genomic control
se1 = S1["se"][j1].astype(np.float64) * gc_fac[BATCH_NAMES[0]]
b2 = S2["beta"][j2].astype(np.float64) * sgn2
se2 = S2["se"][j2].astype(np.float64) * gc_fac[BATCH_NAMES[1]]

M = pd.DataFrame({
    "key": common[g], "ci": (common[g] // KEYMULT).astype(np.int32), "pos": (common[g] % KEYMULT).astype(np.int64),
    "ea": S1["minor"][j1], "nea": S1["major"][j1], "swapped": fl,
    "beta_1": b1.astype(np.float32), "se_1": se1.astype(np.float32), "f_1": f1.astype(np.float32), "n_1": S1["n_inf"][j1], "p_1": 2 * stats.norm.sf(np.abs(b1 / se1)),
    "beta_2": b2.astype(np.float32), "se_2": se2.astype(np.float32), "f_2": f2.astype(np.float32), "n_2": S2["n_inf"][j2], "p_2": 2 * stats.norm.sf(np.abs(b2 / se2)),
})
M["z_1"] = (b1 / se1).astype(np.float32); M["z_2"] = (b2 / se2).astype(np.float32)
M["same_direction"] = np.sign(M.beta_1) == np.sign(M.beta_2)
del b1, se1, b2, se2, f1, f2, z1, z2
print(f"{len(M):,} variants in the meta-analysis table;  memory: {M.memory_usage(deep=True).sum()/1e6:.0f} MB")
display(M.head())

## 9. The Meta-Analysis Itself: Our Own Python Implementation

We now apply the functions of Section 4 to the real data. For every variant we compute

| Column | Meaning |
|---|---|
| `beta_fe`, `se_fe`, `p_fe` | fixed-effect (inverse-variance) estimate, SE and P |
| `Q`, `p_Q`, `I2` | Cochran's Q, its P, and the $I^2$ heterogeneity (fraction) |
| `tau2`, `beta_re`, `se_re`, `p_re` | between-study variance (DerSimonian-Laird) and the random-effects result |
| `z_ss`, `p_ss` | sample-size-weighted *Z*-score (METAL `SCHEME SAMPLESIZE`), weights $\sqrt{n}$ = square root of the number of individuals with data in the batch at that SNP |
| `or_fe`, `or_lo`, `or_hi` | odds ratio of the fixed-effect estimate and its 95 % confidence interval |

In [ ]:
R = meta_analysis(np.vstack([M.beta_1.to_numpy(np.float64), M.beta_2.to_numpy(np.float64)]),
                  np.vstack([M.se_1.to_numpy(np.float64), M.se_2.to_numpy(np.float64)]))
zss, pss = stouffer(np.vstack([M.z_1.to_numpy(np.float64), M.z_2.to_numpy(np.float64)]), np.vstack([M.n_1.to_numpy(np.float64), M.n_2.to_numpy(np.float64)]))
for k in ["beta_fe", "se_fe", "z_fe", "Q", "I2", "tau2", "beta_re", "se_re", "z_re"]:
    M[k] = R[k].astype(np.float32)
for k in ["p_fe", "p_Q", "p_re"]:
    M[k] = R[k]
M["z_ss"] = zss.astype(np.float32); M["p_ss"] = pss
M["w1_fe"] = R["w_fe"][0].astype(np.float32); M["w1_re"] = R["w_re"][0].astype(np.float32)
M["or_fe"] = np.exp(M.beta_fe.astype(np.float64)); M["or_lo"] = np.exp(M.beta_fe.astype(np.float64) - 1.96 * M.se_fe.astype(np.float64)); M["or_hi"] = np.exp(M.beta_fe.astype(np.float64) + 1.96 * M.se_fe.astype(np.float64))
del R

N_META = len(M)
BONF = 0.05 / N_META
print(f"{N_META:,} variants meta-analysed;  Bonferroni threshold 0.05/{N_META:,} = {BONF:.2e}")
sumtab = pd.DataFrame({
    "min P": {"B1 alone": M.p_1.min(), "B2 alone": M.p_2.min(), "fixed effect": M.p_fe.min(), "random effects": M.p_re.min(), "sample-size Z": M.p_ss.min()},
    "lambda_GC": {"B1 alone": lambda_gc(M.p_1), "B2 alone": lambda_gc(M.p_2), "fixed effect": lambda_gc(M.p_fe), "random effects": lambda_gc(M.p_re), "sample-size Z": lambda_gc(M.p_ss)},
    f"SNPs P < {BONF:.1e}": {"B1 alone": int((M.p_1 < BONF).sum()), "B2 alone": int((M.p_2 < BONF).sum()), "fixed effect": int((M.p_fe < BONF).sum()),
                            "random effects": int((M.p_re < BONF).sum()), "sample-size Z": int((M.p_ss < BONF).sum())},
    "SNPs P < 1e-5": {"B1 alone": int((M.p_1 < 1e-5).sum()), "B2 alone": int((M.p_2 < 1e-5).sum()), "fixed effect": int((M.p_fe < 1e-5).sum()),
                      "random effects": int((M.p_re < 1e-5).sum()), "sample-size Z": int((M.p_ss < 1e-5).sum())}})
display(sumtab.style.format({"min P": "{:.2e}", "lambda_GC": "{:.3f}"}))

**Reading the table.** The meta-analysis (fixed effect) should give **smaller p-values than either batch alone** for variants with a consistent effect (the information of the two batches adds up), and the number of SNPs below the threshold is the real test of whether the combination helped. For the heterogeneity-sensitive random-effects model, the p-values are at least as large as the fixed-effect p-values. For two studies the sample-size *Z* and the fixed-effect *Z* are typically very similar when the batches have similar precision.

## 10. The Same Meta-Analysis with GWAMA and METAL

Our Python implementation is transparent, but in practice everybody uses the established programs. They read **one summary-statistics file per study** and align the alleles themselves. We therefore write each batch's file **in its own native orientation** (effect allele = the minor allele *of that batch*, no flipping on our side); if GWAMA and METAL then agree with our Python results, they have aligned the alleles the same way as we did, which also validates our harmonisation.

### 10.1 The input files

One tab-separated file per batch, one row per variant tested in both batches (with compatible alleles):

| Column | Content |
|---|---|
| `MARKERNAME` | the integer key of Section 7.4 (contig index × 10⁹ + position) |
| `EA`, `NEA` | effect allele (this batch's minor allele) and other allele |
| `BETA`, `SE` | effect size (log odds ratio per EA copy) and its standard error |
| `P` | two-sided p-value |
| `N` | number of individuals with data at the SNP |
| `EAF` | effect-allele frequency in the batch |

In [ ]:
ALLELE_LETTER_ARR = ALLELE_LETTER
def write_tool_input(path, key, ea, nea, beta, se, p, n, eaf, step=500_000):
    first = True
    for s in range(0, len(key), step):
        sl = slice(s, s + step)
        pd.DataFrame({"MARKERNAME": key[sl], "EA": ALLELE_LETTER_ARR[ea[sl]], "NEA": ALLELE_LETTER_ARR[nea[sl]], "BETA": beta[sl],
                      "SE": se[sl], "P": p[sl], "N": n[sl], "EAF": eaf[sl]}).to_csv(path, sep="\t", index=False, header=first, mode="w" if first else "a", float_format="%.6g")
        first = False

TOOL_IN = {}
for b, S_, jj, colb in ((BATCH_NAMES[0], S1, j1, "1"), (BATCH_NAMES[1], S2, j2, "2")):
    TOOL_IN[b] = D["summary_stats"] / f"sumstats_{b}.txt"
    if not done(TOOL_IN[b]):
        write_tool_input(TOOL_IN[b], S_["key"][jj], S_["minor"][jj], S_["major"][jj], S_["beta"][jj], M[f"se_{colb}"].to_numpy(), M[f"p_{colb}"].to_numpy(), S_["n_inf"][jj], S_["freq"][jj])
    print(f"{b}: {TOOL_IN[b].name}  ({TOOL_IN[b].stat().st_size/1e6:.1f} MB)")
print(run(f"head -n 4 {TOOL_IN[BATCH_NAMES[0]]}", quiet=True).stdout)

### 10.2 GWAMA

GWAMA reads a **file list** (one study file per line) and the column names. The options we use:

| Option | Meaning |
|---|---|
| `-i filelist` | the list of study files |
| `-o prefix` | output prefix (results in `prefix.out`) |
| `--quantitative` | use `BETA` and `SE` (the default input is odds ratios with confidence intervals). Our `BETA` is a log odds ratio, so this is the right mode. |
| `--random` | also compute the random-effects results |
| `--name_marker`, `--name_ea`, `--name_nea`, `--name_beta`, `--name_se`, `--name_n`, `--name_eaf` | the names of our columns |
| `--genomic_control` | genomic control per study (only if `APPLY_GC`) |

The main output columns are the marker, the reference (effect) and other allele, the pooled frequency, **`beta`, `se`, `z`, `p-value`**, the heterogeneity statistics **`q_p-value`** (P of Cochran's Q) and **`i2`** (in %), the number of studies and samples and the **`effects`** string (`+`/`−` per study = direction of effect).

In [ ]:
GWAMA_BIN = shutil.which("GWAMA") or shutil.which("gwama")
GWAMA_PREFIX = D["gwama"] / "gwama_meta"
GWAMA_OUT = Path(f"{GWAMA_PREFIX}.out")
GW = None
if RUN_GWAMA and GWAMA_BIN:
    filelist = D["gwama"] / "gwama.in"
    filelist.write_text("\n".join(str(TOOL_IN[b]) for b in BATCH_NAMES) + "\n")
    if not done(GWAMA_OUT):
        gc_opt = "--genomic_control" if APPLY_GC else ""
        run(f"{GWAMA_BIN} -i {filelist} -o {GWAMA_PREFIX} --quantitative --random {gc_opt} --name_marker MARKERNAME --name_ea EA --name_nea NEA "
            f"--name_beta BETA --name_se SE --name_n N --name_eaf EAF", log=D["gwama"] / "gwama.log", check=False, show=8)
    if GWAMA_OUT.exists() and GWAMA_OUT.stat().st_size > 0:
        print("\nGWAMA output header:")
        print(run(f"head -n 3 {GWAMA_OUT}", quiet=True).stdout)
        print("rows:", int(run(f"tail -n +2 {GWAMA_OUT} | wc -l", quiet=True).stdout.strip() or 0))
    else:
        print("GWAMA produced no output: see", D["gwama"] / "gwama.log", "(the Python results are not affected).")
else:
    print("GWAMA is not installed or RUN_GWAMA is False: skipping this part (Python results are not affected).")

### 10.3 METAL

METAL is driven by a small **script**, which declares the columns of the input files and the scheme:

```
SCHEME STDERR            # inverse-variance weights (fixed effect)
SEPARATOR TAB
MARKER MARKERNAME
ALLELE EA NEA            # effect allele, other allele
EFFECT BETA
STDERR SE
AVERAGEFREQ ON           # report the average effect-allele frequency
FREQLABEL EAF
PROCESS sumstats_B1.txt  # one PROCESS line per study
PROCESS sumstats_B2.txt
OUTFILE prefix .tbl
ANALYZE HETEROGENEITY    # also compute Q, I2 and the heterogeneity p-value
QUIT
```

The **sample-size scheme** replaces the effect/SE lines by `PVALUE P` and `WEIGHT N`: METAL then combines signed *Z*-scores weighted by $\sqrt N$ (Section 0.6). METAL's output (`.tbl`) has `Allele1`/`Allele2` (the allele that the effect refers to is **`Allele1`**, written in lower case), the combined `Effect`, `StdErr`, `P-value`, a `Direction` string with one symbol per study (`+`, `−`, or `?` if the study has no data for the SNP) and, with heterogeneity analysis, `HetISq`, `HetChiSq`, `HetDf`, `HetPVal`.

In [ ]:
METAL_BIN = shutil.which("metal")
METAL_SE_PREFIX, METAL_SS_PREFIX = D["metal"] / "metal_stderr_", D["metal"] / "metal_samplesize_"
if RUN_METAL and METAL_BIN:
    files = "\n".join(f"PROCESS {TOOL_IN[b]}" for b in BATCH_NAMES)
    gcline = "GENOMICCONTROL ON" if APPLY_GC else "GENOMICCONTROL OFF"
    (D["metal"] / "metal_stderr.txt").write_text(f"SCHEME STDERR\n{gcline}\nSEPARATOR TAB\nMARKER MARKERNAME\nALLELE EA NEA\nEFFECT BETA\nSTDERR SE\n"
                                                 f"AVERAGEFREQ ON\nFREQLABEL EAF\n{files}\nOUTFILE {METAL_SE_PREFIX} .tbl\nANALYZE HETEROGENEITY\nQUIT\n")
    (D["metal"] / "metal_samplesize.txt").write_text(f"SCHEME SAMPLESIZE\n{gcline}\nSEPARATOR TAB\nMARKER MARKERNAME\nALLELE EA NEA\nEFFECT BETA\nPVALUE P\nWEIGHT N\n"
                                                     f"{files}\nOUTFILE {METAL_SS_PREFIX} .tbl\nANALYZE\nQUIT\n")
    for name, pref in (("stderr", METAL_SE_PREFIX), ("samplesize", METAL_SS_PREFIX)):
        if not glob.glob(f"{pref}*.tbl"):
            run(f"{METAL_BIN} {D['metal'] / ('metal_' + name + '.txt')}", log=D["metal"] / f"metal_{name}.log", check=False, show=6)
    for name, pref in (("STDERR", METAL_SE_PREFIX), ("SAMPLESIZE", METAL_SS_PREFIX)):
        t = sorted(glob.glob(f"{pref}*.tbl"))
        if t:
            print(f"\nMETAL {name}: {Path(t[0]).name}"); print(run(f"head -n 3 {t[0]}", quiet=True).stdout)
        else:
            print(f"METAL {name} produced no .tbl file: see the log in", D["metal"])
else:
    print("METAL is not installed or RUN_METAL is False: skipping this part (Python results are not affected).")

### 10.4 Do the programs agree with our Python implementation?

For every variant we compare the *Z*-scores and p-values from GWAMA and METAL with the Python values. Differences in sign are resolved by looking at which allele the tool's effect refers to (our effect allele is the minor allele of batch 1). We expect agreement up to rounding of the printed numbers (six significant digits). Systematic differences would point to a problem with allele alignment or to different conventions (for example genomic control, or how GWAMA treats the standard error).

In [ ]:
def to_key(s):
    return np.rint(pd.to_numeric(s, errors="coerce").to_numpy(np.float64)).astype(np.int64)

def match_rows(keys):
    # index into M for each tool row (M.key is sorted); -1 if absent
    kk = M.key.to_numpy(); idx = np.minimum(np.searchsorted(kk, keys), len(kk) - 1)
    return np.where(kk[idx] == keys, idx, -1)

def find_col(cols, *cands):
    low = {str(c).lower(): c for c in cols}
    for c in cands:
        if c.lower() in low: return low[c.lower()]
    return None

TOOL = {}          # tool results aligned to M (NaN where missing)
cmp_rows = []
EA_LET = ALLELE_LETTER[M.ea.to_numpy()]

if GWAMA_OUT.exists() and GWAMA_OUT.stat().st_size > 0:
    hdr = pd.read_csv(GWAMA_OUT, sep=r"\s+", nrows=0).columns
    c_mark = hdr[0]; c_ref = find_col(hdr, "reference_allele", "EA", "ea"); c_z = find_col(hdr, "z"); c_p = find_col(hdr, "p-value", "p_value", "p")
    c_b = find_col(hdr, "beta"); c_q = find_col(hdr, "q_p-value", "q_p_value"); c_i2 = find_col(hdr, "i2")
    c_re = [c for c in hdr if re.search(r"random|(^|_)re($|_)|_re", str(c).lower())]
    print("GWAMA columns:", list(hdr)); print("random-effects columns detected:", c_re)
    use = [c for c in [c_mark, c_ref, c_z, c_p, c_b, c_q, c_i2] + c_re if c is not None]
    G = pd.read_csv(GWAMA_OUT, sep=r"\s+", usecols=use, dtype={c_mark: str, c_ref: str} if c_ref else {c_mark: str})
    ix = match_rows(to_key(G[c_mark])); ok_ = ix >= 0
    sgn = np.where(G[c_ref].str.upper().to_numpy()[ok_] == EA_LET[ix[ok_]], 1.0, -1.0) if c_ref else 1.0
    gw = pd.DataFrame(index=np.arange(len(M)), dtype=float)
    gw["z"] = np.nan; gw["p"] = np.nan; gw["q_p"] = np.nan; gw["i2"] = np.nan
    if c_z: gw.loc[ix[ok_], "z"] = pd.to_numeric(G[c_z][ok_], errors="coerce").to_numpy() * sgn
    if c_p: gw.loc[ix[ok_], "p"] = pd.to_numeric(G[c_p][ok_], errors="coerce").to_numpy()
    if c_q: gw.loc[ix[ok_], "q_p"] = pd.to_numeric(G[c_q][ok_], errors="coerce").to_numpy()
    if c_i2: gw.loc[ix[ok_], "i2"] = pd.to_numeric(G[c_i2][ok_], errors="coerce").to_numpy() / 100.0
    TOOL["GWAMA"] = gw
    cmp_rows.append(("GWAMA fixed effect", gw["z"].to_numpy(), M.z_fe.to_numpy(np.float64), gw["p"].to_numpy(), M.p_fe.to_numpy()))
    if c_q: cmp_rows.append(("GWAMA heterogeneity (P of Q)", None, None, gw["q_p"].to_numpy(), M.p_Q.to_numpy()))
    del G

t = sorted(glob.glob(f"{METAL_SE_PREFIX}*.tbl"))
if t:
    hdr = pd.read_csv(t[0], sep="\t", nrows=0).columns
    c_m = find_col(hdr, "MarkerName"); c_a1 = find_col(hdr, "Allele1"); c_e = find_col(hdr, "Effect"); c_s = find_col(hdr, "StdErr"); c_pp = find_col(hdr, "P-value")
    c_i = find_col(hdr, "HetISq"); c_hp = find_col(hdr, "HetPVal"); c_d = find_col(hdr, "Direction")
    print("METAL STDERR columns:", list(hdr))
    use = [c for c in [c_m, c_a1, c_e, c_s, c_pp, c_i, c_hp, c_d] if c is not None]
    Mt = pd.read_csv(t[0], sep="\t", usecols=use, dtype={c_m: str, c_a1: str})
    ix = match_rows(to_key(Mt[c_m])); ok_ = ix >= 0
    sgn = np.where(Mt[c_a1].str.upper().to_numpy()[ok_] == EA_LET[ix[ok_]], 1.0, -1.0)
    mt = pd.DataFrame(index=np.arange(len(M)), dtype=float)
    mt["z"] = np.nan; mt["p"] = np.nan; mt["i2"] = np.nan; mt["q_p"] = np.nan
    mt.loc[ix[ok_], "z"] = (pd.to_numeric(Mt[c_e][ok_], errors="coerce") / pd.to_numeric(Mt[c_s][ok_], errors="coerce")).to_numpy() * sgn
    mt.loc[ix[ok_], "p"] = pd.to_numeric(Mt[c_pp][ok_], errors="coerce").to_numpy()
    if c_i: mt.loc[ix[ok_], "i2"] = pd.to_numeric(Mt[c_i][ok_], errors="coerce").to_numpy() / 100.0
    if c_hp: mt.loc[ix[ok_], "q_p"] = pd.to_numeric(Mt[c_hp][ok_], errors="coerce").to_numpy()
    TOOL["METAL"] = mt
    cmp_rows.append(("METAL STDERR (fixed effect)", mt["z"].to_numpy(), M.z_fe.to_numpy(np.float64), mt["p"].to_numpy(), M.p_fe.to_numpy()))
    if c_hp: cmp_rows.append(("METAL heterogeneity (P of Q)", None, None, mt["q_p"].to_numpy(), M.p_Q.to_numpy()))
    del Mt

t = sorted(glob.glob(f"{METAL_SS_PREFIX}*.tbl"))
if t:
    hdr = pd.read_csv(t[0], sep="\t", nrows=0).columns
    c_m = find_col(hdr, "MarkerName"); c_a1 = find_col(hdr, "Allele1"); c_zz = find_col(hdr, "Zscore"); c_pp = find_col(hdr, "P-value")
    print("METAL SAMPLESIZE columns:", list(hdr))
    Ms = pd.read_csv(t[0], sep="\t", usecols=[c for c in [c_m, c_a1, c_zz, c_pp] if c is not None], dtype={c_m: str, c_a1: str})
    ix = match_rows(to_key(Ms[c_m])); ok_ = ix >= 0
    sgn = np.where(Ms[c_a1].str.upper().to_numpy()[ok_] == EA_LET[ix[ok_]], 1.0, -1.0)
    ms = pd.DataFrame(index=np.arange(len(M)), dtype=float); ms["z"] = np.nan; ms["p"] = np.nan
    ms.loc[ix[ok_], "z"] = pd.to_numeric(Ms[c_zz][ok_], errors="coerce").to_numpy() * sgn
    ms.loc[ix[ok_], "p"] = pd.to_numeric(Ms[c_pp][ok_], errors="coerce").to_numpy()
    TOOL["METAL_SS"] = ms
    cmp_rows.append(("METAL SAMPLESIZE", ms["z"].to_numpy(), M.z_ss.to_numpy(np.float64), ms["p"].to_numpy(), M.p_ss.to_numpy()))
    del Ms

if cmp_rows:
    rows = []
    fig, ax = plt.subplots(1, len(cmp_rows), figsize=(4.3 * len(cmp_rows), 4.0), squeeze=False)
    for a, (name, zt, zp, pt, pp) in zip(ax[0], cmp_rows):
        m = np.isfinite(pt) & np.isfinite(pp)
        dz = np.nanmax(np.abs(zt[m] - zp[m])) if zt is not None and m.sum() else np.nan
        rows.append({"comparison": name, "variants compared": int(m.sum()), "of": len(M),
                     "max |Z difference|": dz, "Spearman r of P": stats.spearmanr(pt[m], pp[m])[0] if m.sum() > 2 else np.nan,
                     "max |log10 P difference|": np.nanmax(np.abs(np.log10(np.clip(pt[m], 1e-300, 1)) - np.log10(np.clip(pp[m], 1e-300, 1)))) if m.sum() else np.nan})
        sub = np.random.default_rng(0).choice(np.where(m)[0], min(m.sum(), 30000), replace=False) if m.sum() else []
        a.scatter(-np.log10(np.clip(pp[sub], 1e-300, 1)), -np.log10(np.clip(pt[sub], 1e-300, 1)), s=4, alpha=0.4, color="#009E73", rasterized=True)
        lim = max(a.get_xlim()[1], a.get_ylim()[1]); a.plot([0, lim], [0, lim], "k--", lw=1)
        a.set(xlabel="-log10 P, our Python code", ylabel="-log10 P, program", title=name, ); a.title.set_fontsize(9)
    plt.tight_layout(); savefig("tools_vs_python"); plt.show()
    display(pd.DataFrame(rows).set_index("comparison").style.format({"max |Z difference|": "{:.2e}", "Spearman r of P": "{:.6f}", "max |log10 P difference|": "{:.2e}"}))
else:
    print("Neither GWAMA nor METAL results are available: the analysis below uses our Python implementation, which implements exactly the same formulas.")

**What to expect.** If the alleles were aligned in the same way, the points lie on the diagonal and the Spearman correlation is 1. Small differences (at most a few ×10⁻⁶ in *Z*) come from the six significant digits of the input files. For METAL's heterogeneity p-value the comparison tests our *Q* test against its own. If a program disagrees with Python for a *subset* of variants, look at those variants: they will typically have swapped alleles or be the tri-allelic sites that one of the programs drops.

## 11. The Result: Manhattan and QQ Plots of the Meta-Analysis

The main result is the **fixed-effect (inverse-variance) meta-analysis**, the standard primary analysis (Section 0.5). The **Manhattan plot** (left) shows $-\log_{10}P_{FE}$ for every variant tested in both batches; the red dashed line is Bonferroni for the number of meta-analysed variants. The **QQ plot** (right) compares the observed with the expected distribution of p-values under the null; the fixed-effect, random-effects and sample-size curves are shown together with their genomic inflation factors.

In [ ]:
fig = plt.figure(figsize=(22, 6)); gs = fig.add_gridspec(1, 4)
ax_m = fig.add_subplot(gs[0, :3]); ax_q = fig.add_subplot(gs[0, 3])
manhattan_arrays(ax_m, M.ci.to_numpy(), M.pos.to_numpy(), M.p_fe.to_numpy(), f"Meta-analysis (fixed effect) of {BATCH_NAMES[0]} + {BATCH_NAMES[1]}: {N_META:,} variants", META_COLORS["fixed"], n_tests=N_META)
if GENOME_WIDE_P < 0.05 / N_META:
    ax_m.axhline(-np.log10(GENOME_WIDE_P), color="grey", ls=":", lw=1)
qq_arrays(ax_q, M.p_fe.to_numpy(), META_COLORS["fixed"], label="fixed effect")
qq_arrays(ax_q, M.p_re.to_numpy(), META_COLORS["random"], label="random effects")
qq_arrays(ax_q, M.p_ss.to_numpy(), META_COLORS["ss"], label="sample-size Z")
ax_q.set_title("QQ plot of the meta-analysis")
plt.tight_layout(); savefig("meta_manhattan_qq"); plt.show()

# does meta-analysis add significance compared with the best single batch?
best_single = np.maximum(-np.log10(np.clip(M.p_1.to_numpy(), 1e-300, 1)), -np.log10(np.clip(M.p_2.to_numpy(), 1e-300, 1)))
meta_lp = -np.log10(np.clip(M.p_fe.to_numpy(), 1e-300, 1))
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
sub = np.random.default_rng(3).choice(N_META, min(N_META, 60000), replace=False)
ax[0].scatter(best_single[sub], meta_lp[sub], s=3, alpha=0.3, color="grey", rasterized=True)
top = np.argsort(meta_lp)[::-1][:50]
ax[0].scatter(best_single[top], meta_lp[top], s=14, color=META_COLORS["fixed"], label="top 50 meta-analysis variants")
lim = max(best_single.max(), meta_lp.max()) * 1.03; ax[0].plot([0, lim], [0, lim], "k--", lw=1)
ax[0].set(xlabel="-log10 P of the better batch", ylabel="-log10 P of the meta-analysis", title="Gain from combining the batches\n(above the diagonal = more significant than either batch)"); ax[0].legend(fontsize=8)
inw = in_peak_arr(M.ci.to_numpy(), M.pos.to_numpy())
ax[1].hist(meta_lp[~inw], bins=60, color="grey", alpha=0.7, label="outside the M window"); ax[1].set_yscale("log")
if inw.any(): ax[1].hist(meta_lp[inw], bins=20, color="red", alpha=0.7, label="M window")
ax[1].set(xlabel="-log10 P of the meta-analysis", ylabel="variants", title="Distribution of meta-analysis p-values"); ax[1].legend()
plt.tight_layout(); savefig("meta_gain"); plt.show()
print(f"Variants with P_FE < Bonferroni ({BONF:.1e}): {(M.p_fe < BONF).sum():,}   (in B1 alone: {(M.p_1 < BONF).sum():,};  in B2 alone: {(M.p_2 < BONF).sum():,})")
print(f"Of the variants below Bonferroni in the meta-analysis, {int(inw[(M.p_fe < BONF).to_numpy()].sum()):,} are in the M-locus window on chromosome 26.")

## 12. Top-Ranked Variants, Heterogeneity and Consistency of Direction

### 12.1 Table of the top variants

For the top `TOP_N` variants of the fixed-effect meta-analysis we list everything a reader needs in order to judge them:

* **Per batch**: effect allele frequency, $\hat\beta$, SE, P.
* **Meta-analysis**: fixed-effect odds ratio with its 95 % CI and P; random-effects P; sample-size P.
* **Heterogeneity**: $P_Q$, $I^2$, $\hat\tau^2$.
* **Direction**: whether both batches have the same sign.
* **M-window**: whether the variant is in the M-locus window.

The odds ratio is $e^{\hat\beta}$ per copy of the effect allele (EA); the effect allele is the minor allele of batch 1.

In [ ]:
def variant_table(idx):
    d = M.iloc[idx]
    ci = d.ci.to_numpy()
    out = pd.DataFrame({
        "chr": [CHR_LABEL[fai.contig.iloc[i]] for i in ci], "pos": d.pos.to_numpy(),
        "EA": ALLELE_LETTER[d.ea.to_numpy()], "NEA": ALLELE_LETTER[d.nea.to_numpy()],
        "EAF_B1": d.f_1.to_numpy(), "EAF_B2": d.f_2.to_numpy(),
        "beta_B1": d.beta_1.to_numpy(), "se_B1": d.se_1.to_numpy(), "P_B1": d.p_1.to_numpy(),
        "beta_B2": d.beta_2.to_numpy(), "se_B2": d.se_2.to_numpy(), "P_B2": d.p_2.to_numpy(),
        "same_dir": d.same_direction.to_numpy(), "OR_FE": d.or_fe.to_numpy(), "OR_lo": d.or_lo.to_numpy(), "OR_hi": d.or_hi.to_numpy(),
        "P_FE": d.p_fe.to_numpy(), "P_RE": d.p_re.to_numpy(), "P_sampleZ": d.p_ss.to_numpy(),
        "P_Q": d.p_Q.to_numpy(), "I2_%": 100 * d.I2.to_numpy(), "tau2": d.tau2.to_numpy(),
        "in_M_window": in_peak_arr(ci, d.pos.to_numpy())})
    return out

top_idx = np.argsort(M.p_fe.to_numpy())[:TOP_N]
top_tab = variant_table(top_idx); top_tab.insert(0, "rank", np.arange(1, len(top_tab) + 1))
fmt = {c: "{:.2e}" for c in ["P_B1", "P_B2", "P_FE", "P_RE", "P_sampleZ"]}
fmt.update({c: "{:.3f}" for c in ["EAF_B1", "EAF_B2", "beta_B1", "se_B1", "beta_B2", "se_B2", "OR_FE", "OR_lo", "OR_hi", "P_Q", "tau2"]}); fmt["I2_%"] = "{:.0f}"
display(top_tab.style.format(fmt))
top_tab.to_csv(D["tables"] / "top_variants_meta.tsv", sep="\t", index=False)
M.assign(chr=[CHR_LABEL[fai.contig.iloc[i]] for i in M.ci.to_numpy()]).drop(columns=["w1_fe", "w1_re"]).to_csv(D["tables"] / "meta_results_all_variants.tsv.gz", sep="\t", index=False, float_format="%.6g")
print("tables written to", D["tables"])

**How to read the table.**

* A trustworthy hit has **the same sign in both batches** (`same_dir = True`), non-trivial evidence in each batch (even if each alone is below the threshold), a **non-significant $P_Q$** and a **small $I^2$**, and the random-effects P close to the fixed-effect P. 
* A hit that is driven by **one** batch (one P tiny, the other near 1) has a large $Q$ and $I^2$: its fixed-effect P is not to be trusted at face value, and the random-effects P will be much weaker.
* **Remember $K=2$**: $P_Q$ and $I^2$ come from one degree of freedom. A non-significant $P_Q$ is weak evidence of homogeneity.
* The odds ratios of top SNPs are **approximate** (Section 7.5) and, near perfectly separating loci such as the M locus, can be very large with wide intervals.

### 12.2 Is the direction of effect consistent between the batches?

Under the null hypothesis the two batches are independent, so the effect should point in the same direction at about **50 %** of the variants. For real associations the agreement should be much higher among the **top-ranked** variants. We therefore quantify the concordance

1. **by significance rank:** the fraction of variants with the same sign in both batches among the top 10, 100, 1,000, 10,000 variants of the meta-analysis (with a binomial test against 0.5);
2. **as a discovery/replication exercise:** the variants with $P<10^{-3}$ in batch 1 (*discovery*) are looked up in batch 2 (*replication*): what fraction has the same sign, and what fraction replicates at $P<0.05$ with the same sign? Under the null we expect 50 % and 2.5 %. We also look at the ratio of the replication effect to the discovery effect: a ratio **below 1** is the **winner's curse** (the best hits of a study over-estimate their true effect). The analysis is then repeated with the roles of the batches reversed.

In [ ]:
order = np.argsort(M.p_fe.to_numpy())
sd = M.same_direction.to_numpy()
rows = []
for k in sorted(set([x for x in [10, 100, 1000, 10000] if x < N_META] + [N_META])):
    s = int(sd[order[:k]].sum())
    rows.append({"top variants by P_FE": k if k < N_META else f"all ({N_META:,})", "same direction": s, "fraction": s / k,
                 "binomial P vs 0.5": stats.binomtest(s, k, 0.5).pvalue})
conc = pd.DataFrame(rows).set_index("top variants by P_FE")
display(conc.style.format({"fraction": "{:.3f}", "binomial P vs 0.5": "{:.2e}"}))

def discovery_replication(a, b, thr=1e-3, min_n=10):
    pa, pb = M[f"p_{a}"].to_numpy(), M[f"p_{b}"].to_numpy(); ba, bb = M[f"beta_{a}"].to_numpy(np.float64), M[f"beta_{b}"].to_numpy(np.float64)
    sel = pa < thr; label = f"P_{a} < {thr:g}"
    if sel.sum() < min_n:
        sel = np.zeros(len(pa), bool); sel[np.argsort(pa)[:100]] = True; label = f"top 100 by P_{a}"
    same_ = np.sign(ba[sel]) == np.sign(bb[sel]); repl = same_ & (pb[sel] < 0.05)
    ratio = np.median((bb[sel] * np.sign(ba[sel])) / np.abs(ba[sel]))
    return {"discovery": f"B{a}", "replication": f"B{b}", "selection": label, "n discovery": int(sel.sum()), "same direction": same_.mean(),
            "binomial P": stats.binomtest(int(same_.sum()), int(sel.sum()), 0.5).pvalue, "replicated (P<0.05, same dir.)": repl.mean(), "expected under null": 0.025,
            "median beta_repl / beta_disc": ratio}
dr = pd.DataFrame([discovery_replication("1", "2"), discovery_replication("2", "1")])
display(dr.style.format({"same direction": "{:.3f}", "binomial P": "{:.2e}", "replicated (P<0.05, same dir.)": "{:.3f}", "median beta_repl / beta_disc": "{:.2f}"}))
conc.to_csv(D["tables"] / "direction_concordance_by_rank.tsv", sep="\t"); dr.to_csv(D["tables"] / "discovery_replication.tsv", sep="\t", index=False)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
# (a) concordance as a function of rank
ks = np.unique(np.round(np.geomspace(10, N_META, 25)).astype(int))
fr = np.array([sd[order[:k]].mean() for k in ks])
lo = np.array([stats.binomtest(int(sd[order[:k]].sum()), k, 0.5).proportion_ci(0.95, method="wilson").low for k in ks])
hi = np.array([stats.binomtest(int(sd[order[:k]].sum()), k, 0.5).proportion_ci(0.95, method="wilson").high for k in ks])
ax[0].plot(ks, fr, "o-", color=META_COLORS["fixed"]); ax[0].fill_between(ks, lo, hi, color=META_COLORS["fixed"], alpha=0.2)
ax[0].axhline(0.5, color="k", ls="--", lw=1); ax[0].set_xscale("log"); ax[0].set_ylim(0.3, 1.02)
ax[0].set(xlabel="number of top variants (ranked by P of the meta-analysis)", ylabel="fraction with the same direction in B1 and B2", title="Direction consistency by rank\n(0.5 = no agreement beyond chance)")
# (b) Z1 vs Z2 for the top variants
topn = order[:500]
col = np.where(sd[topn], "#009E73", "#D55E00")
ax[1].scatter(M.z_1.to_numpy()[topn], M.z_2.to_numpy()[topn], s=12, c=col)
ax[1].axhline(0, c="k", lw=.6); ax[1].axvline(0, c="k", lw=.6)
ax[1].set(xlabel=f"Z in {BATCH_NAMES[0]}", ylabel=f"Z in {BATCH_NAMES[1]} (harmonised)", title="Top 500 variants: green = same direction, orange = opposite")
# (c) heterogeneity
ax[2].hist(M.p_Q.dropna(), bins=20, color="grey", alpha=0.8, density=True)
ax[2].axhline(1, color="k", ls="--", lw=1)
ax[2].set(xlabel="P of Cochran's Q", ylabel="density", title=f"Heterogeneity across all {N_META:,} variants\n(uniform = no excess heterogeneity; median I² = {100*np.nanmedian(M.I2):.0f}%)")
plt.tight_layout(); savefig("direction_heterogeneity"); plt.show()

het = pd.DataFrame({"all variants": [N_META, (M.p_Q < 0.05).mean(), 100 * np.nanmedian(M.I2), (M.I2 > 0.5).mean(), M.tau2.mean()],
                    "top 100 by P_FE": [100, (M.p_Q.to_numpy()[order[:100]] < 0.05).mean(), 100 * np.nanmedian(M.I2.to_numpy()[order[:100]]), (M.I2.to_numpy()[order[:100]] > 0.5).mean(), M.tau2.to_numpy()[order[:100]].mean()]},
                   index=["variants", "fraction with P_Q < 0.05", "median I2 (%)", "fraction with I2 > 50%", "mean tau2"])
display(het.style.format("{:.3f}"))

**Reading the three panels.**

* **(a)** If the batches carry no shared signal the curve hugs 0.5; for real associations it rises towards 1 as we move to the top-ranked variants. The shaded band is the 95 % confidence interval of the fraction.
* **(b)** Variants in the upper-right and lower-left quadrants (green) are consistent; orange points in the other quadrants are inconsistent and are the ones with large $Q$.
* **(c)** A flat histogram means that the observed heterogeneity is no more than expected from sampling noise. An enrichment of small $P_Q$ values means that the batches really differ (population, depth, technical effects) for a subset of variants.

## 13. Forest Plots

A **forest plot** shows, for **one variant**, the effect estimate of each study with its 95 % confidence interval (a square, sized by the study's weight in the fixed-effect analysis) and the pooled estimates (diamonds: green = fixed effect, pink = random effects). It answers at a glance: *do the studies agree, how precise is each, and does the pooled effect exclude no effect* (the dashed line at OR = 1)?

How to read it:

* intervals that **overlap** and sit on the same side of 1: consistent effect;
* intervals on **opposite sides** of 1: heterogeneity; the fixed-effect diamond is then misleading;
* a **wide diamond** (pink) compared with the green one signals heterogeneity: the random-effects model widens the interval;
* the **size of the square** is the weight: the more precise study dominates.

We show the **top 6 variants** and, if it is not among them, the most significant variant in the **M-locus window** (the positive control). The x-axis is on the **odds-ratio scale** (log axis). The odds ratios are score-based approximations (Section 7.5).

In [ ]:
sel_idx = list(np.argsort(M.p_fe.to_numpy())[:6])
inw_all = in_peak_arr(M.ci.to_numpy(), M.pos.to_numpy())
if inw_all.any():
    best_m = int(np.where(inw_all)[0][np.argmin(M.p_fe.to_numpy()[inw_all])])
    if best_m not in sel_idx: sel_idx.append(best_m)
ncol = 3; nrow = int(np.ceil(len(sel_idx) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.4 * ncol, 3.0 * nrow), squeeze=False)
for a in axes.ravel()[len(sel_idx):]: a.axis("off")
for a, i in zip(axes.ravel(), sel_idx):
    r = M.iloc[i]
    lab = f"chr {CHR_LABEL[fai.contig.iloc[int(r.ci)]]}:{int(r.pos):,}  {ALLELE_LETTER[int(r.ea)]}/{ALLELE_LETTER[int(r.nea)]}"
    forest_panel(a, BATCH_NAMES, [r.beta_1, r.beta_2], [r.se_1, r.se_2], fe=(r.beta_fe, r.se_fe), re=(r.beta_re, r.se_re),
                 title=f"{lab}\nP_FE={r.p_fe:.1e}, P_Q={r.p_Q:.2f}, I²={100*r.I2:.0f}%" + ("  [M window]" if in_peak_arr(np.array([int(r.ci)]), np.array([int(r.pos)]))[0] else ""),
                 log_or=True, study_colors=BATCH_COLORS)
plt.tight_layout(); savefig("forest_top_variants"); plt.show()

### 13.1 One forest plot for many variants

The same information for the top 12 variants in one figure: three rows per variant (batch 1, batch 2, fixed-effect pooled estimate), which makes it easy to scan for variants where the two batches point in different directions.

In [ ]:
top12 = np.argsort(M.p_fe.to_numpy())[:12]
fig, ax = plt.subplots(figsize=(8.5, 0.62 * 3 * len(top12) + 1.2))
y = 0.0; yt, yl = [], []
for j, i in enumerate(top12[::-1]):
    r = M.iloc[i]
    for lab, b_, s_, col, mk in ((BATCH_NAMES[1], r.beta_2, r.se_2, BATCH_COLORS[BATCH_NAMES[1]], "s"), (BATCH_NAMES[0], r.beta_1, r.se_1, BATCH_COLORS[BATCH_NAMES[0]], "s"),
                                 ("fixed effect", r.beta_fe, r.se_fe, META_COLORS["fixed"], "D")):
        ax.plot([np.exp(b_ - 1.96 * s_), np.exp(b_ + 1.96 * s_)], [y, y], color="k", lw=1.2)
        ax.scatter([np.exp(b_)], [y], s=50, marker=mk, color=col, zorder=3)
        yt.append(y); yl.append(lab); y += 1
    ax.text(1.01, y - 2, f"chr{CHR_LABEL[fai.contig.iloc[int(r.ci)]]}:{int(r.pos):,}  P={r.p_fe:.0e}", transform=ax.get_yaxis_transform(), fontsize=7, va="center")
    y += 0.6
ax.set_yticks(yt); ax.set_yticklabels(yl, fontsize=7)
ax.axvline(1, color="grey", ls="--"); ax.set_xscale("log"); ax.set_xlabel("odds ratio per effect allele (log scale)")
ax.set_title("Top 12 variants of the meta-analysis"); ax.set_ylim(-0.8, y)
plt.tight_layout(); savefig("forest_top12"); plt.show()

## 14. Discussion: What Did We Learn?

Use the tables and figures above to answer these questions (the answers depend on your data, which is why they are posed as questions).

1. **Did the meta-analysis help?** Compare the number of variants below the Bonferroni threshold and the smallest p-values in B1, B2 and the fixed-effect meta-analysis (Section 9 table, Section 11). Is the M locus (red points) more significant after combining?
2. **Are the two single-batch analyses consistent?** Do the Manhattan plots point to the same region? How large is the correlation of *Z*-scores between the batches, and how does the fraction of same-direction variants change with rank (Section 12.2)?
3. **Heterogeneity.** What fraction of variants has $P_Q<0.05$? Is the distribution of $P_Q$ flat? Is the top variant heterogeneous? Remember that with $K=2$ the test has 1 degree of freedom.
4. **Fixed vs random effects.** Where do $P_{FE}$ and $P_{RE}$ differ? Are those variants also the heterogeneous ones (Section 9/12.1)?
5. **Do GWAMA, METAL and the Python code agree?** If not, where do they differ and why (Section 10.4)?
6. **Winner's curse.** In the discovery/replication table, is the median ratio of replication to discovery effect below 1?
7. **Calibration.** What are the $\lambda_{GC}$ values of B1, B2 and the meta-analysis? Would genomic control (`APPLY_GC = True`) change your conclusions?
8. **Low coverage.** How would the results change if the two batches had different sequencing depths (`SPLIT_MODE = "balance_depth"` vs a deliberately unbalanced split)? What would that do to heterogeneity?

### 14.1 Limitations of this experiment

* Two random halves of **one sample** are **not independent cohorts**: they share the population, the sequencing protocol and the reference genome, so we expect **homogeneous** true effects. Real meta-analyses combine cohorts that differ in all of these, and heterogeneity is then a central issue.
* **10 individuals per batch** is tiny. With 20 chromosomes, allele frequencies are very noisy, score tests are approximate, and `-minCount` and `-minMaf` remove most variants. The conclusions are a demonstration of the method, not of the biology.
* The **effect sizes** are score-based approximations; for strong effects they are biased towards zero. Reporting a proper maximum-likelihood odds ratio would require a per-SNP logistic regression on the dosages (or ANGSD's latent genotype model, `-doAsso 4`).
* **Variants tested in one batch only** are not meta-analysed here. In real meta-analyses they are retained with $K=1$ and flagged.

### 14.2 Exercises

1. Set `SPLIT_SEED` to another value and re-run from Section 5. How stable is the top hit list? How often is the M locus found by each batch alone and by the meta-analysis? (Delete the cached `stats_B*.npz` files, or set `FORCE = True`.)
2. Change `N_PER_MORPH_PER_BATCH` (up to half of the number of individuals of each morph) and see how power changes.
3. Set `APPLY_GC = True`. Which p-values change most?
4. Set `EFFECT_SOURCE = "score"` and compare the top variants with the default.
5. Make the batches **unbalanced in depth** (swap the deep individuals into one batch) and look at heterogeneity.
6. Add a third study (for example the other 5 + 5 individuals if you have more than 20). How do $I^2$ and the power of $Q$ change when $K=3$?

### 14.3 References

* Willer CJ, Li Y, Abecasis GR (2010). **METAL: fast and efficient meta-analysis of genomewide association scans.** *Bioinformatics* 26:2190–2191.
* Mägi R, Morris AP (2010). **GWAMA: software for genome-wide association meta-analysis.** *BMC Bioinformatics* 11:288.
* DerSimonian R, Laird N (1986). Meta-analysis in clinical trials. *Controlled Clinical Trials* 7:177–188.
* Higgins JPT, Thompson SG (2002). Quantifying heterogeneity in a meta-analysis. *Statistics in Medicine* 21:1539–1558.
* Skotte L, Korneliussen TS, Albrechtsen A (2012). Association testing for next-generation sequencing data using score statistics. *Genetic Epidemiology* 36:430–437.
* Korneliussen TS, Albrechtsen A, Nielsen R (2014). ANGSD: Analysis of Next Generation Sequencing Data. *BMC Bioinformatics* 15:356.
* Devlin B, Roeder K (1999). Genomic control for association studies. *Biometrics* 55:997–1004.
* Evangelou E, Ioannidis JPA (2013). Meta-analysis methods for genome-wide association studies and beyond. *Nature Reviews Genetics* 14:379–389.

## 15. Files Created

Everything is on your Drive in `metaanalysis_HypMis/` next to the `bams/` folder. The next cell lists the most important files.

In [ ]:
deliver = {
    "batch assignment": D["tables"] / "batch_assignment.tsv",
    "ANGSD association, batch 1": next(iter(sorted(glob.glob(f"{ASSO[BATCH_NAMES[0]]}.lrt*.gz"))), ""),
    "ANGSD association, batch 2": next(iter(sorted(glob.glob(f"{ASSO[BATCH_NAMES[1]]}.lrt*.gz"))), ""),
    "summary statistics, batch 1": TOOL_IN[BATCH_NAMES[0]], "summary statistics, batch 2": TOOL_IN[BATCH_NAMES[1]],
    "GWAMA results": GWAMA_OUT, "METAL results (STDERR)": next(iter(sorted(glob.glob(f"{METAL_SE_PREFIX}*.tbl"))), ""),
    "METAL results (SAMPLESIZE)": next(iter(sorted(glob.glob(f"{METAL_SS_PREFIX}*.tbl"))), ""),
    "meta-analysis, all variants": D["tables"] / "meta_results_all_variants.tsv.gz", "top variants": D["tables"] / "top_variants_meta.tsv",
    "direction concordance": D["tables"] / "direction_concordance_by_rank.tsv", "discovery/replication": D["tables"] / "discovery_replication.tsv",
    "Manhattan, two batches": D["figures"] / "manhattan_two_batches.png", "Manhattan + QQ, meta-analysis": D["figures"] / "meta_manhattan_qq.png",
    "forest plots": D["figures"] / "forest_top_variants.png"}
for k, v in deliver.items():
    v = Path(v) if v else None
    print(f"{k:34s} {'OK ' if v and v.exists() else '-- '} {v if v else ''}")

## 16. Summary

In this notebook we

✅ installed **Conda, ANGSD, SAMtools, GWAMA and METAL** in Colab and mounted Google Drive;  
✅ split the 20 butterflies into **two balanced batches** (5 *m* + 5 *i* each), checked their sequencing-depth balance, and ran **ANGSD genotype-likelihood calling and association testing separately** in each batch (two Manhattan plots side by side, with QQ plots and λ);  
✅ converted the ANGSD score test into **summary statistics** (effect allele, β, SE, P, N, frequency) and validated them against ANGSD's own LRT;  
✅ **harmonised the alleles** of the two studies (flipping swapped alleles, dropping incompatible ones);  
✅ implemented **fixed-effect (inverse-variance), random-effects (DerSimonian-Laird) and sample-size-weighted *Z* meta-analysis** in Python, learned their properties from a simulation, and cross-validated them with **GWAMA** and **METAL**;  
✅ assessed **heterogeneity** (Cochran's *Q*, *I*², τ²) and the **consistency of effect directions** (by rank, and as discovery/replication with the winner's-curse check);  
✅ produced the final **Manhattan and QQ plot of the meta-analysis**, a table of the **top-ranked variants**, and **forest plots**.

**The central ideas to take away:** meta-analysis adds the information of independent studies; the allele orientation must be harmonised before anything is added; the fixed-effect model assumes one true effect, which heterogeneity statistics test; and a hit that is **consistent in direction across independent samples** is far more believable than one driven by a single batch.